In [178]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
from os import listdir
from os.path import isfile, join
import re
import MasterPSD as PSD
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import glob
import os, re, csv
from typing import Optional, List, Dict, Any
from datetime import datetime, timezone
from typing import Tuple, Dict, Any, List, Optional
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
from os import listdir
from os.path import isfile, join
import re
import MasterPSD as PSD
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import VisAnalysis as Vis

try:
    from numba import njit
    NUMBA_OK = True
except Exception:
    NUMBA_OK = False

def _plot_with_side_hist(x, y, xlabel, ylabel, ylim, bins=50, title=None):
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec
    import numpy as np

    fig = plt.figure(figsize=(10, 4))
    gs = GridSpec(1, 2, width_ratios=[4, 1], wspace=0.05)

    ax = fig.add_subplot(gs[0, 0])
    ax_hist = fig.add_subplot(gs[0, 1], sharey=ax)

    ax.plot(x, y, lw=1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.25)
    if title:
        ax.set_title(title)

    finite = np.isfinite(y)
    ax_hist.hist(
        y[finite],
        bins=bins,
        orientation="horizontal",
        density=True,
        alpha=0.7,
    )
    ax_hist.grid(True, alpha=0.25)
    ax_hist.set_xlabel("PDF")

    plt.setp(ax_hist.get_yticklabels(), visible=False)
    ax_hist.tick_params(axis="y", length=0)

    plt.tight_layout()
    plt.show()

def _to_seconds(ts: np.ndarray) -> np.ndarray:
    ts = np.asarray(ts)
    if np.issubdtype(ts.dtype, np.datetime64):
        t0 = ts[0]
        return (ts - t0) / np.timedelta64(1, "s")
    return ts.astype(float) - float(ts[0])


def _phase_from_ramp(t_s: np.ndarray, f_ramp: float, phase_offset: float = 0.0) -> np.ndarray:
    return 2.0 * np.pi * f_ramp * t_s + phase_offset


if NUMBA_OK:
    @njit(cache=True)
    def _prefix_sum(arr):
        out = np.empty(arr.size + 1, dtype=np.float64)
        out[0] = 0.0
        s = 0.0
        for i in range(arr.size):
            s += arr[i]
            out[i + 1] = s
        return out

    @njit(cache=True)
    def _win_sum(ps, a, b):
        return ps[b] - ps[a]

    @njit(cache=True)
    def _poisson_nll(y, lam, lam_floor=1e-12):
        nll = 0.0
        for i in range(y.size):
            li = lam[i]
            if li < lam_floor:
                li = lam_floor
            nll += li - y[i] * np.log(li)
        return nll

    @njit(cache=True)
    def _fit_alpha_beta_for_delta(yw, cw, iters=6, lam_floor=1e-6):
        """
        For fixed delta, we have cw = cos(phi+delta).
        Fit lambda = alpha + beta*cw with Poisson NLL using a few Newton-ish steps
        on (alpha,beta) with constraints alpha>0, beta>=0, and alpha>|beta| to keep lambda positive-ish.

        Returns alpha, beta, nll.
        """
        # Initial guesses
        ymean = 0.0
        for i in range(yw.size):
            ymean += yw[i]
        ymean /= yw.size

        alpha = max(ymean, lam_floor)
        beta  = 0.0

        # Iterative reweighted least squares-like updates for identity mean
        for _ in range(iters):
            # weights w = 1/lambda
            sw0 = 0.0
            sw1 = 0.0
            sw2 = 0.0
            swy0 = 0.0
            swy1 = 0.0

            for i in range(yw.size):
                lam = alpha + beta * cw[i]
                if lam < lam_floor:
                    lam = lam_floor
                w = 1.0 / lam

                x1 = cw[i]
                sw0 += w
                sw1 += w * x1
                sw2 += w * x1 * x1

                yi = yw[i]
                swy0 += w * yi
                swy1 += w * yi * x1

            # Solve 2x2: [[sw0, sw1],[sw1, sw2]] [alpha,beta] = [swy0,swy1]
            det = sw0 * sw2 - sw1 * sw1
            if abs(det) < 1e-18:
                break

            alpha_new = ( sw2 * swy0 - sw1 * swy1) / det
            beta_new  = (-sw1 * swy0 + sw0 * swy1) / det

            # Constraints (keep stable)
            if alpha_new < lam_floor:
                alpha_new = lam_floor
            # nonnegative beta (visibility amplitude)
            if beta_new < 0.0:
                beta_new = 0.0
            # keep alpha >= beta to avoid negative lambda
            if beta_new > alpha_new - lam_floor:
                beta_new = max(0.0, alpha_new - lam_floor)

            alpha, beta = alpha_new, beta_new

        # compute nll
        lam = np.empty(yw.size, dtype=np.float64)
        for i in range(yw.size):
            li = alpha + beta * cw[i]
            if li < lam_floor:
                li = lam_floor
            lam[i] = li

        nll = _poisson_nll(yw, lam, lam_floor=lam_floor)
        return alpha, beta, nll

    @njit(cache=True)
    def _mle_phasegrid_visibility(
        y, cphi, sphi,
        K, S, n_min,
        deltas, iters_ab=6, lam_floor=1e-6
    ):
        """
        Per window: grid search over delta.
        For each delta:
          cw = cos(phi+delta) = cphi*cosd - sphi*sind
          Fit alpha,beta (Poisson) and compute NLL
        Choose best delta => V=beta/alpha
        """
        n = y.size
        nwin = (n - K) // S + 1
        V_out = np.empty(nwin, dtype=np.float64)
        delta_out = np.empty(nwin, dtype=np.float64)

        for w in range(nwin):
            start = w * S
            end = start + K

            # total counts gate
            sy = 0.0
            for i in range(start, end):
                sy += y[i]
            if sy < n_min:
                V_out[w] = np.nan
                delta_out[w] = np.nan
                continue

            best_nll = 1e300
            best_V = np.nan
            best_d = 0.0

            for di in range(deltas.size):
                d = deltas[di]
                cd = np.cos(d)
                sd = np.sin(d)

                # build cw = cos(phi+d) quickly from precomputed cos/sin(phi)
                cw = np.empty(K, dtype=np.float64)
                yw = np.empty(K, dtype=np.float64)
                idx = 0
                for i in range(start, end):
                    cw[idx] = cphi[i] * cd - sphi[i] * sd
                    yw[idx] = y[i]
                    idx += 1

                alpha, beta, nll = _fit_alpha_beta_for_delta(
                    yw, cw, iters=iters_ab, lam_floor=lam_floor
                )

                if nll < best_nll:
                    best_nll = nll
                    best_d = d
                    best_V = beta / alpha if alpha > 0 else np.nan

            V_out[w] = best_V
            delta_out[w] = best_d

        return V_out, delta_out


from typing import Optional, Tuple, Dict, Any

def lockin_visibility_vs_time_mle_numba(
    ts,
    counts,
    f_ramp: float,
    tau: float,
    step: Optional[float] = None,
    phase_offset: float = 0.0,
    n_min: float = 0.0,
    clip: bool = True,
    n_phase: int = 64,
    iters_ab: int = 8,
    lam_floor: float = 1e-6,
    verbose: bool = True,
    make_plots: bool = True,
    # --- NEW ---
    pm_volts: Optional[np.ndarray] = None,
    V2pi_volts: Optional[float] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
    """
    Numba-friendly Poisson MLE-like visibility vs time.
    If pm_volts is provided, phase is computed from PM voltage via V2pi_volts.
    Otherwise, phase is computed from a continuous ramp model using f_ramp.

    Returns t_centers, V(t), QBER(t), diagnostics
    """
    if not NUMBA_OK:
        raise ImportError("numba is not available. Install with: pip install numba")

    t_s = _to_seconds(np.asarray(ts)).astype(np.float64)
    y = np.clip(np.nan_to_num(np.asarray(counts), nan=0.0), 0, None).astype(np.float64)
    n = min(t_s.size, y.size)
    t_s = t_s[:n]
    y = y[:n]
    if n < 3:
        raise ValueError("Need at least 3 samples.")

    dts = np.diff(t_s)
    dt_pos = dts[dts > 0]
    if dt_pos.size == 0:
        raise ValueError("Timestamps not strictly increasing.")
    dt = float(np.mean(dt_pos))

    if step is None:
        step = tau / 2.0
    K = max(3, int(round(tau / dt)))
    S = max(1, int(round(step / dt)))
    if K > n:
        return np.array([]), np.array([]), np.array([]), {"dt": dt, "K": K, "S": S, "n_windows": 0}

    # ---- Phase model (NEW)
    using_pm = (pm_volts is not None)
    if using_pm:
        if V2pi_volts is None:
            raise ValueError("If pm_volts is provided, you must also provide V2pi_volts.")
        pv = np.asarray(pm_volts, dtype=float)
        pv = pv[:n]  # align to counts
        phi = _phase_from_pm_volts(pv, float(V2pi_volts), phase_offset=phase_offset).astype(np.float64)
    else:
        phi = _phase_from_ramp(t_s, f_ramp=f_ramp, phase_offset=phase_offset).astype(np.float64)

    cphi = np.cos(phi).astype(np.float64)
    sphi = np.sin(phi).astype(np.float64)

    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False).astype(np.float64)

    V_out, delta_out = _mle_phasegrid_visibility(
        y, cphi, sphi,
        K, S, float(n_min),
        deltas, iters_ab=int(iters_ab), lam_floor=float(lam_floor)
    )

    nwin = V_out.size
    centers_idx = (np.arange(nwin, dtype=np.int64) * S) + (K // 2)
    t_centers = t_s[centers_idx]

    Vq = np.clip(V_out, 0.0, 1.0) if clip else V_out
    QBER = 0.5 * (1.0 - Vq)

    diagnostics = {
        "dt": dt,
        "K_samples": int(K),
        "S_samples": int(S),
        "tau_eff_s": float(K * dt),
        "step_eff_s": float(S * dt),
        "n_windows_total": int(nwin),
        "n_windows_used": int(np.sum(np.isfinite(V_out))),
        "n_phase": int(n_phase),
        "iters_ab": int(iters_ab),
        "lam_floor": float(lam_floor),
        "n_min_counts": float(n_min),
        "clip_for_qber": bool(clip),
        "phase_source": "pm_volts" if using_pm else "time_ramp",
        "V2pi_volts": float(V2pi_volts) if using_pm else None,
    }

    if verbose:
        print("[Numba phase-grid Poisson MLE visibility]")
        for k, v in diagnostics.items():
            print(f"  {k}: {v}")

    if make_plots and t_centers.size:
        _plot_with_side_hist(
            t_centers,
            V_out,
            xlabel="Time (s)",
            ylabel=f"V_MLE_NUMBA(t)  (tau ≈ {K*dt*1e3:.2f} ms)",
            ylim=(-0.05, 1.05 if clip else 1.3),
            bins=60,
            title="Visibility vs Time (Poisson MLE, Numba)",
        )
        _plot_with_side_hist(
            t_centers,
            QBER,
            xlabel="Time (s)",
            ylabel="QBER(t)",
            ylim=(0.0, 0.5),
            bins=60,
            title="QBER vs Time (from Visibility, Poisson MLE, Numba)",
        )

    return t_centers, V_out, QBER, diagnostics

In [180]:
import numpy as np
import matplotlib.pyplot as plt

def plot_phase_fit_curve(yw, phiw, a, b, d0, lam_floor=1e-12, n_curve=600, n_bins=12):
    """
    yw: counts in the window (shape K)
    phiw: phase in the window (shape K), can be unwrapped or not
    a,b: fitted alpha, beta
    d0: chosen delta
    """

    # ---- Phase wrapping for plotting
    phase_wrapped = np.mod(phiw + d0, 2*np.pi)
    order = np.argsort(phase_wrapped)
    
    phase_sorted = phase_wrapped[order]
    counts_sorted = yw[order]
    
    # Model prediction
    cw = np.cos(phiw + d0)
    lam = np.maximum(a + b*cw, lam_floor)
    lam_sorted = lam[order]
    
    # ---- 1σ Poisson errors (model-based)
    sigma = np.sqrt(np.maximum(lam_sorted, lam_floor))
    
    # ---- Smooth model curve
    phi_dense = np.linspace(0, 2*np.pi, 500)
    lam_dense = a + b * np.cos(phi_dense)
    
    fig = new_appendix_figure()
    ax = fig.add_subplot(1, 1, 1)
    
    # Raw counts with 1σ error bars
    ax.errorbar(
        phase_sorted,
        counts_sorted,
        yerr=sigma,
        fmt='s',
        markersize=7,
        alpha=0.5,
        label="Counts (1σ Poisson)",
        elinewidth=2.3
    )
    
    # Smooth model curve
    ax.plot(phi_dense, lam_dense, lw=2.6, label="Poisson MLE fit")
    
    ax.set_title(f"Fitted interference model (V ≈ {b/max(a,1e-12):.3f})")
    ax.set_xlabel("Phase (rad)")
    ax.set_ylabel("Counts")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right', fontsize=14, frameon=True)
    #ax.set_ylim(0,7)
    fig.tight_layout()
    plt.savefig('AppendixA3_MLE_FittedModel.pdf', dpi=300)
    plt.show()


In [ ]:
# ---- Phase wrapping for plotting
phase_wrapped = np.mod(phiw + d0, 2*np.pi)
order = np.argsort(phase_wrapped)

phase_sorted = phase_wrapped[order]
counts_sorted = yw[order]

# Model prediction
cw = np.cos(phiw + d0)
lam = np.maximum(a + b*cw, lam_floor)
lam_sorted = lam[order]

# ---- 1σ Poisson errors (model-based)
sigma = np.sqrt(np.maximum(lam_sorted, lam_floor))

# ---- Smooth model curve
phi_dense = np.linspace(0, 2*np.pi, 500)
lam_dense = a + b * np.cos(phi_dense)

fig = new_appendix_figure()
ax = fig.add_subplot(1, 1, 1)

# Raw counts with 1σ error bars
ax.errorbar(
    phase_sorted,
    counts_sorted,
    yerr=sigma,
    fmt='s',
    markersize=18,
    alpha=0.45,
    label="Counts (1σ Poisson)"
)

# Smooth model curve
ax.plot(phi_dense, lam_dense, lw=2.2, label="Poisson MLE fit")

ax.set_title(f"Fitted interference model (V ≈ {b/max(a,1e-12):.3f})")
ax.set_xlabel("Phase (rad)")
ax.set_ylabel("Counts")
ax.grid(True, alpha=0.25)
ax.legend(loc='upper right', fontsize=14, frameon=True)

fig.tight_layout()
plt.savefig('AppendixA3_MLE_FittedModel.pdf', dpi=300)
plt.show()

In [196]:
import numpy as np
import matplotlib.pyplot as plt

def set_appendix_plot_style():
    plt.rcParams.update({'font.size': 10})
    plt.rc('axes', titlesize=14)
    plt.rc('axes', labelsize=13)

def new_appendix_figure():
    fig = plt.figure()
    fig.set_size_inches(8, 6)
    return fig

import matplotlib as mpl

def draw_lockin_window_arrows(
    ax,
    t: np.ndarray,
    K: int,
    S: int,
    *,
    rows: int = 2,
    top_margin: float = 0.08,
    cmap_name: str = "tab20",
    arrowstyle: str = "<->",
    arrow_lw: float = 1.6,
    arrow_alpha: float = 0.95,
    vline_alpha: float = 0.9,
    draw_end_lines: bool = True,
    label: bool = True,
    fontsize: int = 13,
    max_windows: int = 12,     # keep the diagram uncluttered
):
    """
    Welch-like overlay for lock-in sliding windows:
      window i spans indices [i*S, i*S + K)

    Returns:
      starts (sample indices), and diagnostics dict.
    """
    t = np.asarray(t, dtype=float)
    N = t.size
    if N < 2 or K < 2 or S < 1:
        return np.array([], dtype=int), {"K": K, "S": S, "colors": []}

    nwin_total = (N - K) // S + 1
    nwin_show = int(min(max_windows, max(0, nwin_total)))
    starts = (np.arange(nwin_show, dtype=int) * S)

    # colormap (new API + fallback)
    try:
        cmap = mpl.colormaps.get_cmap(cmap_name)
    except AttributeError:
        from matplotlib.cm import get_cmap as _legacy_get_cmap
        cmap = _legacy_get_cmap(cmap_name)

    y_min, y_max = ax.get_ylim()
    yr = (y_max - y_min)
    y_base = y_max - top_margin * yr
    row_offsets = np.linspace(0.0, 0.06 * yr, max(1, rows))

    colors = []
    for i, s in enumerate(starts):
        a = t[s]
        e = s + K - 1
        if e >= N:
            break
        b = t[e]

        color = cmap((2 * i) % cmap.N)
        colors.append(color)

        # boundary lines (grey dashed, as in Welch diagram)
        ax.axvline(a, color='grey', linestyle="--", linewidth=1.1, alpha=vline_alpha * 0.5)
        if draw_end_lines:
            ax.axvline(b, color='grey', linestyle="--", linewidth=1.1, alpha=vline_alpha * 0.5)

        # arrow row staggering
        row = i % max(1, rows)
        y = y_base - row_offsets[row]

        ax.annotate(
            "", xy=(b, y), xytext=(a, y),
            arrowprops=dict(arrowstyle=arrowstyle, lw=arrow_lw, color=color, alpha=arrow_alpha)
        )

        if label:
            ax.text((a + b) / 2, y, f"{i}", ha="center", va="bottom",
                    fontsize=fontsize, color=color)

    ax.set_ylim(y_min, y_max)
    return starts, {
        "K_samples": int(K),
        "S_samples": int(S),
        "nwin_total": int(nwin_total),
        "nwin_shown": int(nwin_show),
        "colors": colors,
    }

def _phase_from_pm_volts(pm_volts: np.ndarray, V2pi_volts: float, phase_offset: float = 0.0) -> np.ndarray:
    """
    Convert PM drive voltage to optical phase (radians), assuming linear response:
        phi = 2π * (V / V2pi) + phase_offset
    """
    v = np.asarray(pm_volts, dtype=float)
    if V2pi_volts <= 0:
        raise ValueError("V2pi_volts must be > 0.")
    return (2.0 * np.pi * (v / float(V2pi_volts))) + float(phase_offset)

def _get_phase_reference(
    ts: np.ndarray,
    f_ramp: float,
    phase_offset: float = 0.0,
    pm_volts: Optional[np.ndarray] = None,
    V2pi_volts: Optional[float] = None,
) -> Tuple[np.ndarray, str]:
    """
    Return phase array and a human-readable description of its origin.

    Priority:
      1) pm_volts + V2pi_volts (hardware-accurate)
      2) ideal linear ramp from time

    Returns
    -------
    phi : np.ndarray
        Phase in radians.
    source : str
        Description of phase source (for plot titles / captions).
    """
    t_s = _to_seconds(ts)

    if pm_volts is not None:
        if V2pi_volts is None:
            raise ValueError("V2pi_volts must be provided when pm_volts is used.")
        pv = np.asarray(pm_volts, dtype=float)[:t_s.size]
        phi = (2.0 * np.pi * pv / float(V2pi_volts)) + float(phase_offset)
        source = "Phase from PM voltage (staircase ramp)"
    else:
        phi = (2.0 * np.pi * float(f_ramp) * t_s) + float(phase_offset)
        source = "Phase from ideal linear time ramp"

    return phi.astype(np.float64), source


def _wrap_to_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def _side_hist(ax, data, bins=60, range_=None):
    """Draw a normalized sideways histogram in an inset axis to the right of ax."""
    data = np.asarray(data)
    data = data[np.isfinite(data)]
    if data.size == 0:
        return

    # inset at right
    inset = ax.inset_axes([1.02, 0.05, 0.22, 0.90], transform=ax.transAxes)
    counts, edges = np.histogram(data, bins=bins, range=range_, density=True)
    centers = 0.5*(edges[:-1] + edges[1:])
    inset.plot(counts, centers, linewidth=1)
    inset.set_xlabel("pdf", fontsize=9)
    inset.tick_params(labelsize=8)
    inset.grid(True, alpha=0.2)
    # hide y labels (shared visually with main axis)
    inset.set_yticklabels([])

def _plot_windows_overlay(t, y, K, S, n_show=6, title="Windowing illustration"):
    """Show first few sliding windows like a Welch segmentation diagram."""
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(t, y, linewidth=1)

    for w in range(n_show):
        start = w*S
        end = start + K
        if end >= y.size:
            break
        ax.axvline(t[start], linestyle="--", linewidth=1)
        ax.axvline(t[end-1], linestyle="--", linewidth=1)
        ax.text(t[start], np.nanmax(y)*0.95, f"{w}", fontsize=10)

    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Counts")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

def plot_counts_and_phase_combined(
    t: np.ndarray,
    counts: np.ndarray,
    phi: np.ndarray,
    *,
    phase_src: str = "phase ramp",
    max_points: int = 20000,
):
    plt.rcParams.update({'font.size': 10})
    plt.rc('axes', titlesize=14)
    plt.rc('axes', labelsize=13) 
    n = t.size
    ds = max(1, int(np.ceil(n / max_points)))

    fig, ax = plt.subplots(
        2, 1,gridspec_kw={'height_ratios': [2, 1]}
    )
    fig.set_size_inches(8, 6)

    # -------- A.1 Raw counts --------
    ax[0].plot(t[::ds], counts[::ds], lw=1)
    ax[0].set_ylabel("Counts")
    ax[0].set_title("Raw photon counts vs time /w PM Ramp")
    ax[0].grid(False)
    ax[0].set_ylim(0,np.max(counts[::ds])+0.5)
    ax[0].set_xticklabels([])

    # -------- A.2 Phase reference --------
    phi_wrapped = np.mod(phi, 2*np.pi)
    ax[1].plot(t[::ds], phi_wrapped[::ds],color='tab:orange', lw=0.8)
    ax[1].set_ylabel("Phase (rad)")
    ax[1].set_xlabel("Time (s)")
    ax[1].grid(False)

    # ✅ Phase ticks: 0, π, 2π
    ax[1].set_yticks([0, np.pi, 2*np.pi])
    ax[1].set_yticklabels([r"$0$", r"$\pi$", r"$2\pi$"])
    ax[1].set_ylim(0,2*np.pi + 0.1)

    # ✅ Remove excess x-axis space
    ax[0].set_xlim(t[0], t[-1])
    ax[1].set_xlim(t[0], t[-1])
    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.045)
    #fig.set_tight_layout(True)
    plt.savefig('AppendixA3_MLE_Raw.pdf', dpi=300)
    plt.show()


def appendix_plots_lockin_mle_pipeline(
    ts,
    counts,
    f_ramp,
    tau,
    step=None,
    phase_offset=0.0,
    n_min=0.0,
    n_phase=64,
    iters_ab=8,
    lam_floor=1e-6,
    pm_volts=None,
    V2pi_volts=None,
    window_index=10,
    max_points=20000,
    max_windows_overlay=12,
):

    # ----- enforce style
    set_appendix_plot_style()

    # ----- prepare data
    t = _to_seconds(ts).astype(float)
    y = np.clip(np.nan_to_num(np.asarray(counts, float), nan=0.0), 0, None)
    n = min(t.size, y.size)
    t = t[:n]
    y = y[:n]

    dts = np.diff(t)
    dt_pos = dts[dts > 0]
    dt = float(np.mean(dt_pos)) if dt_pos.size else float(np.mean(dts))

    if step is None:
        step = tau / 2.0
    K = max(3, int(np.round(tau / dt)))
    S = max(1, int(np.round(step / dt)))

    # If user intended no overlap, enforce it exactly
    if np.isclose(step, tau, rtol=0, atol=0.5*dt):
        S = K

    # ----- phase reference (auto)
    phi, phase_src = _get_phase_reference(
        ts, f_ramp=float(f_ramp), phase_offset=float(phase_offset),
        pm_volts=pm_volts, V2pi_volts=V2pi_volts
    )
    phi = phi[:n]
    cphi = np.cos(phi)
    sphi = np.sin(phi)

    # downsample index for big datasets (only for display of raw traces)
    ds = max(1, int(np.ceil(n / max_points)))

    plot_counts_and_phase_combined(
        t=t,
        counts=y,
        phi=phi,
        phase_src=phase_src,
        )

    # ========= A.2 Segmented counts plot with Welch-style τ windows =========
    fig = new_appendix_figure()
    ax = fig.add_subplot(1, 1, 1)
    ax.plot(t[::ds], y[::ds], lw=1)
    ax.set_title(f"Sliding τ-windows (overlap = {1 - S/max(K,1):.0%}, K={K} samples)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Counts")
    ax.grid(False)
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(0, ymax + 1)
    ax.set_xlim(np.min(t[::ds]),np.max(t[::ds]))
    # IMPORTANT: overlay needs correct ylim, so call after plotting
    _starts, overlay_diag = draw_lockin_window_arrows(
        ax, t, K=K, S=S,
        rows=2,
        top_margin=0.08,
        cmap_name="tab20",
        arrowstyle="<->",
        arrow_lw=1.6,
        arrow_alpha=0.95,
        vline_alpha=0.9,
        draw_end_lines=False,
        label=True,
        fontsize=13,
        max_windows=max_windows_overlay,
        )

    fig.tight_layout()
    plt.savefig('AppendixA3_MLE_Windows.pdf', dpi=300)
    plt.show()

    # ========= A.3 Per-window NLL vs delta (one chosen window) =========
    w = int(window_index)
    start = w * S
    end = start + K
    if end >= n:
        raise ValueError("window_index too large for this (tau, step, data length).")

    yw = y[start:end]
    phiw = phi[start:end]

    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False)
    nlls = np.empty(deltas.size, dtype=float)

    for i, d in enumerate(deltas):
        cw = np.cos(phiw + d)
        a, b, nll = _fit_alpha_beta_for_delta(yw, cw, iters=int(iters_ab), lam_floor=float(lam_floor))
        nlls[i] = nll

    best = int(np.argmin(nlls))
    d0 = float(deltas[best])

    fig = new_appendix_figure()
    ax = fig.add_subplot(1, 1, 1)
    ax.plot(deltas, nlls, lw=1.5)
    ax.axvline(d0, ls="--", lw=1.2)
    ax.set_title("Per-window Poisson negative log-likelihood vs phase offset δ")
    ax.set_xlabel("δ (rad)")
    ax.set_ylabel("NLL")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.savefig('AppendixA3_MLE_PoiNNL.pdf', dpi=300)
    plt.show()

    # ========= A.4 Model fit in phase domain (one window) =========
    cw = np.cos(phiw + d0)
    a, b, _ = _fit_alpha_beta_for_delta(yw, cw, iters=int(iters_ab), lam_floor=float(lam_floor))
    
    Kwin = yw.size
    n_bins = int(np.clip(np.sqrt(Kwin), 8, 16))   # 8–16 is a good visual range
    plot_phase_fit_curve(yw, phiw, a, b, d0, lam_floor=lam_floor, n_bins=n_bins)

    # ========= A.5 Final V(t), QBER(t) =========
    tcent, Vt, Q, diag = lockin_visibility_vs_time_mle_numba(
        ts, counts,
        f_ramp=float(f_ramp),
        tau=float(tau),
        step=float(step),
        phase_offset=float(phase_offset),
        n_min=float(n_min),
        clip=False,
        n_phase=int(n_phase),
        iters_ab=int(iters_ab),
        lam_floor=float(lam_floor),
        verbose=False,
        make_plots=False,
        pm_volts=pm_volts,
        V2pi_volts=V2pi_volts,
    )

    ds2 = max(1, int(np.ceil(tcent.size / 5000)))

    VisWin=Vt[window_index]
    TWin=tcent[window_index]
    QBWin=Q[window_index]
    
    fig = new_appendix_figure()
    ax = fig.add_subplot(1, 1, 1)
    ax.plot(tcent[::ds2], Vt[::ds2], marker='s',markersize=12, linestyle='None')
    ax.plot(TWin,VisWin, marker='.',markersize=18, label='τ-window 2')
    ax.set_title("Estimated visibility vs time")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Visibility")
    ax.grid(True, alpha=0.25)
    ax.set_xlim(0,np.max(tcent[::ds2]))
    ax.set_ylim(0,1)
    ax.legend(loc='upper right', fontsize="14", frameon=True)
    fig.tight_layout()
    plt.savefig('AppendixA3_MLE_Vis.pdf', dpi=300)
    plt.show()

    fig = new_appendix_figure()
    ax = fig.add_subplot(1, 1, 1)
    ax.plot(tcent[::ds2], Q[::ds2], marker='s',markersize=12, linestyle='None')
    ax.plot(TWin,QBWin, marker='.',markersize=18, label='τ-window 2')
    ax.set_title("Estimated QBER vs time (from visibility)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("QBER")
    ax.grid(True, alpha=0.25)
    ax.set_xlim(0,np.max(tcent[::ds2]))
    ax.set_ylim(0,0.5)
    ax.legend(loc='upper right', fontsize="14", frameon=True)
    fig.tight_layout()
    plt.savefig('AppendixA3_MLE_QBER.pdf', dpi=300)
    plt.show()
    
    # Print diag summary
    print("\n[MLE visibility diagnostics]")
    for k, v in diag.items():
        print(f"  {k}: {v}")
    print("\n[Window overlay diagnostics]")
    for k, v in overlay_diag.items():
        print(f"  {k}: {v}")

    return tcent, Vt, Q, diag


In [198]:
DNF = Vis.load_measurements(base_dir="D:\PhD Thesis Writing\Data", date_str="2025-11-26", subfolder="Day2LongNoFibre", verbose=True)

Loaded 12 measurements from D:\PhD Thesis Writing\Data\2025-11-26\Day2LongNoFibre


In [200]:
ts = DNF[4]['data']['timestamps'][0:300]
counts = DNF[4]['data']['counts'][0:300]
pm_volts = DNF[4]['data']['pm_volts'][0:300]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])
f_ramp=1000.0
tau=3e-3

In [202]:
DNF[4]['data']['pm_V2pi_volts']

array(6.86)

In [204]:
%matplotlib qt
t_cent, Vt, Q, diag = appendix_plots_lockin_mle_pipeline(
    ts,
    counts,
    f_ramp,
    tau,
    step=tau,
    phase_offset=0.0,
    n_min=0.0,
    n_phase=64,
    iters_ab=8,
    lam_floor=1e-6,
    pm_volts=pm_volts,
    V2pi_volts=V2pi,
    window_index=2,
    max_points=10000,
    max_windows_overlay=16
)


[MLE visibility diagnostics]
  dt: 0.0001
  K_samples: 30
  S_samples: 30
  tau_eff_s: 0.003
  step_eff_s: 0.003
  n_windows_total: 10
  n_windows_used: 10
  n_phase: 64
  iters_ab: 8
  lam_floor: 1e-06
  n_min_counts: 0.0
  clip_for_qber: False
  phase_source: pm_volts
  V2pi_volts: 6.86

[Window overlay diagnostics]
  K_samples: 30
  S_samples: 30
  nwin_total: 10
  nwin_shown: 10
  colors: [(0.12156862745098039, 0.4666666666666667, 0.7058823529411765, 1.0), (1.0, 0.4980392156862745, 0.054901960784313725, 1.0), (0.17254901960784313, 0.6274509803921569, 0.17254901960784313, 1.0), (0.8392156862745098, 0.15294117647058825, 0.1568627450980392, 1.0), (0.5803921568627451, 0.403921568627451, 0.7411764705882353, 1.0), (0.5490196078431373, 0.33725490196078434, 0.29411764705882354, 1.0), (0.8901960784313725, 0.4666666666666667, 0.7607843137254902, 1.0), (0.4980392156862745, 0.4980392156862745, 0.4980392156862745, 1.0), (0.7372549019607844, 0.7411764705882353, 0.13333333333333333, 1.0), (0.090

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _wrap_to_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def _fit_ab_python(yw, cw, iters=8, lam_floor=1e-6):
    """
    Pure-Python mirror of _fit_alpha_beta_for_delta (for plotting/diagnostics).
    Fits lambda = alpha + beta*cw with Poisson-ish IRLS updates + constraints.
    """
    ymean = float(np.mean(yw))
    alpha = max(ymean, lam_floor)
    beta  = 0.0

    for _ in range(int(iters)):
        lam = alpha + beta * cw
        lam = np.maximum(lam, lam_floor)
        wts = 1.0 / lam

        sw0  = float(np.sum(wts))
        sw1  = float(np.sum(wts * cw))
        sw2  = float(np.sum(wts * cw * cw))
        swy0 = float(np.sum(wts * yw))
        swy1 = float(np.sum(wts * yw * cw))

        det = sw0 * sw2 - sw1 * sw1
        if abs(det) < 1e-18:
            break

        alpha_new = ( sw2 * swy0 - sw1 * swy1) / det
        beta_new  = (-sw1 * swy0 + sw0 * swy1) / det

        # Constraints (match your njit logic)
        if alpha_new < lam_floor:
            alpha_new = lam_floor
        if beta_new < 0.0:
            beta_new = 0.0
        if beta_new > alpha_new - lam_floor:
            beta_new = max(0.0, alpha_new - lam_floor)

        alpha, beta = float(alpha_new), float(beta_new)

    lam = np.maximum(alpha + beta * cw, lam_floor)
    nll = float(np.sum(lam - yw * np.log(lam)))  # + const ignored
    return alpha, beta, nll


def diagnose_step4_window(
    ts,
    counts,
    f_ramp: float,
    tau: float,
    step: float = None,
    window_index: int = 10,
    phase_offset: float = 0.0,
    n_min: float = 0.0,
    n_phase: int = 64,
    iters_ab: int = 8,
    lam_floor: float = 1e-6,
    show_param_curves: bool = True,
):
    """
    Step-4 diagnostic plots for ONE sliding window in lockin_visibility_vs_time_mle_numba.

    Produces:
      1) window counts vs time
      2) counts vs wrapped phase
      3) NLL vs delta (grid search)
      4) fitted lambda vs phase overlay (best delta)
      5) (optional) alpha/beta/V vs delta
    """
    # --- preprocess
    t_s = _to_seconds(np.asarray(ts)).astype(np.float64)
    y = np.clip(np.nan_to_num(np.asarray(counts), nan=0.0), 0, None).astype(np.float64)
    n = min(t_s.size, y.size)
    t_s = t_s[:n]
    y   = y[:n]
    if n < 3:
        raise ValueError("Need at least 3 samples.")

    dts = np.diff(t_s)
    dt_pos = dts[dts > 0]
    if dt_pos.size == 0:
        raise ValueError("Timestamps not strictly increasing.")
    dt = float(np.mean(dt_pos))

    if step is None:
        step = tau / 2.0
    K = max(3, int(round(tau / dt)))
    S = max(1, int(round(step / dt)))
    if K > n:
        raise ValueError("tau too large: window longer than dataset.")

    # --- pick window
    w = int(window_index)
    start = w * S
    end = start + K
    if end > n:
        raise ValueError("window_index too large for this tau/step/dataset length.")

    yw = y[start:end].copy()
    tw = t_s[start:end].copy()

    sy = float(np.sum(yw))
    if sy < n_min:
        print(f"[Window {w}] skipped: sum(yw)={sy:.3f} < n_min={n_min}")
        return None

    # --- phase model
    phi = _phase_from_ramp(t_s, f_ramp=f_ramp, phase_offset=phase_offset).astype(np.float64)
    phiw = phi[start:end].copy()

    # --- delta grid
    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False).astype(np.float64)

    nlls  = np.full(deltas.size, np.nan, dtype=float)
    alphas = np.full(deltas.size, np.nan, dtype=float)
    betas  = np.full(deltas.size, np.nan, dtype=float)
    Vs     = np.full(deltas.size, np.nan, dtype=float)

    # --- grid search (Step 4)
    for i, d in enumerate(deltas):
        cw = np.cos(phiw + d)
        a, b, nll = _fit_ab_python(yw, cw, iters=iters_ab, lam_floor=lam_floor)
        alphas[i] = a
        betas[i]  = b
        nlls[i]   = nll
        Vs[i]     = (b / a) if a > 0 else np.nan

    best_i = int(np.nanargmin(nlls))
    best_d = float(deltas[best_i])
    best_a = float(alphas[best_i])
    best_b = float(betas[best_i])
    best_V = float(Vs[best_i])

    # ========= PLOTS =========
    # 1) counts vs time in window
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.plot(tw, yw, linewidth=1)
    ax.set_title(f"Step 4A: Windowed counts vs time (window={w}, K={K}, S={S}, sum={sy:.1f})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Counts")
    ax.grid(True, alpha=0.25)
    plt.show()

    # 2) counts vs wrapped phase in window
    ph_wrap = _wrap_to_pi(phiw + best_d)  # show after best alignment
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.scatter(ph_wrap, yw, s=12, alpha=0.7)
    ax.set_title("Step 4B: Counts vs wrapped phase in window (after applying best δ)")
    ax.set_xlabel("Wrapped phase (rad)")
    ax.set_ylabel("Counts")
    ax.grid(True, alpha=0.25)
    plt.show()

    # 3) NLL vs delta
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.plot(deltas, nlls, linewidth=1)
    ax.axvline(best_d, linestyle="--", linewidth=1)
    ax.set_title(f"Step 4C: Poisson NLL vs δ (best δ={best_d:+.3f} rad)")
    ax.set_xlabel("δ (rad)")
    ax.set_ylabel("NLL (lower is better)")
    ax.grid(True, alpha=0.25)
    plt.show()

    # 4) fit overlay in phase domain
    # sort by phase for a clean curve
    cw_best = np.cos(phiw + best_d)
    lam_fit = np.maximum(best_a + best_b * cw_best, lam_floor)
    ph_fit = _wrap_to_pi(phiw + best_d)
    idx = np.argsort(ph_fit)

    fig, ax = plt.subplots(figsize=(9, 3.4), constrained_layout=True)
    ax.scatter(ph_fit[idx], yw[idx], s=12, alpha=0.6, label="Observed counts")
    ax.plot(ph_fit[idx], lam_fit[idx], linewidth=2,
            label=r"Fitted $\lambda=\alpha+\beta\cos(\phi+\delta)$")
    ax.set_title(f"Step 4D: Best fit in phase domain (α={best_a:.3g}, β={best_b:.3g}, V=β/α={best_V:.4f})")
    ax.set_xlabel("Wrapped phase (rad)")
    ax.set_ylabel("Counts / expected counts")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    # 5) optional parameter curves vs delta
    if show_param_curves:
        fig, ax = plt.subplots(figsize=(9, 3.4), constrained_layout=True)
        ax.plot(deltas, Vs, linewidth=1, label="V(δ)=β/α")
        ax.axvline(best_d, linestyle="--", linewidth=1)
        ax.set_title("Step 4E: Visibility candidate vs δ")
        ax.set_xlabel("δ (rad)")
        ax.set_ylabel("V(δ)")
        ax.grid(True, alpha=0.25)
        ax.legend()
        plt.show()

        fig, ax = plt.subplots(figsize=(9, 3.4), constrained_layout=True)
        ax.plot(deltas, alphas, linewidth=1, label="α(δ)")
        ax.plot(deltas, betas,  linewidth=1, label="β(δ)")
        ax.axvline(best_d, linestyle="--", linewidth=1)
        ax.set_title("Step 4F: Fitted α and β vs δ")
        ax.set_xlabel("δ (rad)")
        ax.set_ylabel("Parameter value")
        ax.grid(True, alpha=0.25)
        ax.legend()
        plt.show()

    # return useful debug info
    return {
        "window_index": w,
        "start_idx": start,
        "end_idx": end,
        "dt": dt,
        "K": K,
        "S": S,
        "tau_eff_s": K * dt,
        "step_eff_s": S * dt,
        "sum_counts": sy,
        "best_delta": best_d,
        "best_alpha": best_a,
        "best_beta": best_b,
        "best_V": best_V,
        "deltas": deltas,
        "nlls": nlls,
        "alphas": alphas,
        "betas": betas,
        "Vs": Vs,
        "t_window": tw,
        "y_window": yw,
        "phi_window": phiw,
    }


In [ ]:
import numpy as np

def irls_fit_alpha_beta_stepwise(
    yw,
    cw,
    iters=8,
    lam_floor=1e-6,
    verbose=True
):
    """
    Step-by-step version of your _fit_alpha_beta_for_delta.
    Fits lambda = alpha + beta*cw with Poisson-ish IRLS updates.

    Prints iteration trace: alpha, beta, V, det, NLL.
    """
    yw = np.asarray(yw, dtype=float)
    cw = np.asarray(cw, dtype=float)
    assert yw.shape == cw.shape
    K = yw.size

    # Initial guesses (same as your code)
    alpha = max(float(np.mean(yw)), lam_floor)
    beta = 0.0

    def poisson_nll(y, lam):
        lam = np.maximum(lam, lam_floor)
        return float(np.sum(lam - y*np.log(lam)))

    if verbose:
        lam0 = np.maximum(alpha + beta*cw, lam_floor)
        print(f"init: alpha={alpha:.6g}, beta={beta:.6g}, V={beta/alpha if alpha>0 else np.nan:.6g}, NLL={poisson_nll(yw, lam0):.6g}")

    history = []

    for k in range(iters):
        # Build weights w = 1/lambda
        lam = alpha + beta*cw
        lam = np.maximum(lam, lam_floor)
        w = 1.0 / lam

        # Accumulate weighted normal-equation sums
        sw0  = float(np.sum(w))
        sw1  = float(np.sum(w*cw))
        sw2  = float(np.sum(w*cw*cw))
        swy0 = float(np.sum(w*yw))
        swy1 = float(np.sum(w*yw*cw))

        det = sw0*sw2 - sw1*sw1
        if abs(det) < 1e-18:
            if verbose:
                print(f"iter {k+1}: det ~ 0 (ill-conditioned), stopping.")
            break

        # Solve 2x2 for new alpha,beta
        alpha_new = ( sw2*swy0 - sw1*swy1) / det
        beta_new  = (-sw1*swy0 + sw0*swy1) / det

        # Constraints (match your updated code)
        if alpha_new < lam_floor:
            alpha_new = lam_floor
        if beta_new < 0.0:
            beta_new = 0.0
        if beta_new > alpha_new - lam_floor:
            beta_new = max(0.0, alpha_new - lam_floor)

        alpha, beta = float(alpha_new), float(beta_new)

        lam_fit = np.maximum(alpha + beta*cw, lam_floor)
        nll = poisson_nll(yw, lam_fit)
        V = beta/alpha if alpha > 0 else np.nan

        history.append((alpha, beta, V, det, nll))

        if verbose:
            print(
                f"iter {k+1}: alpha={alpha:.6g}, beta={beta:.6g}, V={V:.6g}, "
                f"det={det:.3e}, NLL={nll:.6g}, lam_min={lam_fit.min():.6g}, lam_max={lam_fit.max():.6g}"
            )

    return alpha, beta, history


# ---- Example usage with a synthetic "window" ----
if __name__ == "__main__":
    np.random.seed(0)

    # pretend we already chose a delta, so cw is known
    K = 20
    cw = np.cos(np.linspace(-np.pi, np.pi, K, endpoint=False))

    # generate synthetic Poisson counts from a known alpha,beta
    alpha_true = 3.0
    beta_true  = 2.4  # V_true = 0.8
    lam_true = alpha_true + beta_true*cw
    yw = np.random.poisson(lam_true)

    print("yw:", yw.astype(int))
    print("cw:", np.round(cw, 3))

    alpha_hat, beta_hat, hist = irls_fit_alpha_beta_stepwise(yw, cw, iters=10, lam_floor=1e-6, verbose=True)
    print("\nfinal:", alpha_hat, beta_hat, "V=", beta_hat/alpha_hat)


In [ ]:
dbg = diagnose_step4_window(
    ts, counts,
    f_ramp=1000.0,
    tau=tau,
    step=tau,
    window_index=2,
    n_phase=64,
    iters_ab=8,
    lam_floor=1e-6,
    n_min=0,
    show_param_curves=True,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Helpers (match your logic)
# -----------------------------
def to_seconds(ts):
    ts = np.asarray(ts)
    if np.issubdtype(ts.dtype, np.datetime64):
        t0 = ts[0]
        return (ts - t0) / np.timedelta64(1, "s")
    return ts.astype(float) - float(ts[0])

def wrap_to_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def poisson_nll(y, lam, lam_floor=1e-12):
    lam = np.maximum(lam, lam_floor)
    return float(np.sum(lam - y*np.log(lam)))  # +const ignored

def irls_fit_alpha_beta_trace(yw, cw, iters=8, lam_floor=1e-6):
    """
    Step-by-step IRLS-like solver: lambda = alpha + beta*cw
    Constraints: alpha>=lam_floor, beta>=0, beta<=alpha-lam_floor
    Returns (alpha, beta, nll, trace)
    trace: list of dicts with alpha,beta,V,nll,det,lam_min,lam_max
    """
    yw = np.asarray(yw, dtype=float)
    cw = np.asarray(cw, dtype=float)

    alpha = max(float(np.mean(yw)), lam_floor)
    beta = 0.0
    trace = []

    for k in range(int(iters)):
        lam = alpha + beta * cw
        lam = np.maximum(lam, lam_floor)
        w = 1.0 / lam

        sw0  = float(np.sum(w))
        sw1  = float(np.sum(w * cw))
        sw2  = float(np.sum(w * cw * cw))
        swy0 = float(np.sum(w * yw))
        swy1 = float(np.sum(w * yw * cw))

        det = sw0 * sw2 - sw1 * sw1
        if abs(det) < 1e-18:
            # ill-conditioned (usually means poor phase coverage in window)
            trace.append({
                "iter": k+1, "alpha": alpha, "beta": beta, "V": (beta/alpha if alpha>0 else np.nan),
                "nll": poisson_nll(yw, lam, lam_floor), "det": det,
                "lam_min": float(lam.min()), "lam_max": float(lam.max()),
                "note": "det ~ 0, stopped"
            })
            break

        alpha_new = ( sw2 * swy0 - sw1 * swy1 ) / det
        beta_new  = (-sw1 * swy0 + sw0 * swy1 ) / det

        # constraints
        if alpha_new < lam_floor:
            alpha_new = lam_floor
        if beta_new < 0.0:
            beta_new = 0.0
        if beta_new > alpha_new - lam_floor:
            beta_new = max(0.0, alpha_new - lam_floor)

        alpha, beta = float(alpha_new), float(beta_new)
        lam = np.maximum(alpha + beta*cw, lam_floor)
        nll = poisson_nll(yw, lam, lam_floor)
        trace.append({
            "iter": k+1,
            "alpha": alpha,
            "beta": beta,
            "V": (beta/alpha if alpha>0 else np.nan),
            "nll": nll,
            "det": det,
            "lam_min": float(lam.min()),
            "lam_max": float(lam.max()),
        })

    # final
    lam = np.maximum(alpha + beta*cw, lam_floor)
    nll = poisson_nll(yw, lam, lam_floor)
    return alpha, beta, nll, trace

def grid_search_delta(yw, phi_w, n_phase=64, iters_ab=8, lam_floor=1e-6):
    """
    For a fixed window:
      try deltas, fit alpha,beta with IRLS, compute NLL, return best.
    """
    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False)
    nlls = np.full(deltas.size, np.nan)
    alphas = np.full(deltas.size, np.nan)
    betas = np.full(deltas.size, np.nan)
    Vs = np.full(deltas.size, np.nan)

    for i, d in enumerate(deltas):
        cw = np.cos(phi_w + d)
        a, b, nll, _ = irls_fit_alpha_beta_trace(yw, cw, iters=iters_ab, lam_floor=lam_floor)
        alphas[i] = a
        betas[i] = b
        nlls[i] = nll
        Vs[i] = (b/a) if a > 0 else np.nan

    best_i = int(np.nanargmin(nlls))
    return {
        "deltas": deltas,
        "nlls": nlls,
        "alphas": alphas,
        "betas": betas,
        "Vs": Vs,
        "best_i": best_i,
        "best_delta": float(deltas[best_i]),
        "best_alpha": float(alphas[best_i]),
        "best_beta": float(betas[best_i]),
        "best_V": float(Vs[best_i]),
    }

# -----------------------------
# Main diagnostic you asked for
# -----------------------------
def show_window_mle_diagnostics(
    ts,
    counts,
    pm_volts,
    V2pi,
    f_ramp=1000.0,
    tau=3e-3,
    step=None,
    window_index=0,
    n_phase=64,
    iters_ab=8,
    lam_floor=1e-6,
    use_pm_phase=True,   # True: use pm_volts→phase, False: use ideal ramp from t
):
    """
    Produces:
      - window selection / dt / K
      - NLL vs delta
      - V vs delta
      - counts vs wrapped phase (best delta)
      - IRLS iteration trace for best delta
    """
    t_s = to_seconds(ts).astype(np.float64)
    y = np.asarray(counts, dtype=float)
    pm = np.asarray(pm_volts, dtype=float)

    n = min(t_s.size, y.size, pm.size)
    t_s = t_s[:n]; y = y[:n]; pm = pm[:n]
    if n < 3:
        raise ValueError("Need >=3 samples.")

    dt = float(np.mean(np.diff(t_s)))
    if step is None:
        step = tau/2.0
    K = max(3, int(round(tau/dt)))
    S = max(1, int(round(step/dt)))

    start = window_index * S
    end = start + K
    if end > n:
        raise ValueError("window_index too large for this tau/step/data length.")

    yw = y[start:end]
    tw = t_s[start:end]

    # phase model
    if use_pm_phase:
        # map PM volts to phase; unwrap to make it continuous, then we can wrap for plots
        phi = 2*np.pi * (pm / float(V2pi))
        phi = np.unwrap(phi)  # important: avoids sawtooth wrap artifacts in fitting
        phase_label = "PM volts → phase (unwrapped)"
    else:
        phi = 2*np.pi * float(f_ramp) * t_s
        phase_label = "Ideal ramp phase 2π f t"

    phi_w = phi[start:end]

    print(f"[Window diagnostics]")
    print(f"  use_phase_model: {phase_label}")
    print(f"  n_samples total: {n}")
    print(f"  dt ≈ {dt*1e6:.2f} µs  (Fs ≈ {1/dt/1e3:.2f} kHz)")
    print(f"  tau_req={tau:.3e} s -> K={K} samples -> tau_eff={K*dt:.3e} s")
    print(f"  step_req={step:.3e} s -> S={S} samples -> step_eff={S*dt:.3e} s")
    print(f"  window_index={window_index} => idx [{start}:{end})")
    print(f"  sum counts in window: {float(np.sum(yw)):.1f}, mean/bin: {float(np.mean(yw)):.3f}")

    # Plot windowed counts vs time
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(tw, yw, linewidth=1)
    ax.set_title("Windowed counts vs time")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("counts")
    ax.grid(True, alpha=0.25)
    plt.show()

    # Grid search over delta
    gs = grid_search_delta(yw, phi_w, n_phase=n_phase, iters_ab=iters_ab, lam_floor=lam_floor)
    d = gs["deltas"]; nlls = gs["nlls"]; Vs = gs["Vs"]
    best_d = gs["best_delta"]; best_V = gs["best_V"]
    best_a = gs["best_alpha"]; best_b = gs["best_beta"]

    # Plot NLL vs delta
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(d, nlls, linewidth=1)
    ax.axvline(best_d, linestyle="--", linewidth=1)
    ax.set_title(f"NLL vs δ (best δ={best_d:+.3f} rad)")
    ax.set_xlabel("δ (rad)")
    ax.set_ylabel("Poisson NLL")
    ax.grid(True, alpha=0.25)
    plt.show()

    # Plot V vs delta
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(d, Vs, linewidth=1)
    ax.axvline(best_d, linestyle="--", linewidth=1)
    ax.set_title(f"V(δ)=β/α vs δ (best V={best_V:.4f})")
    ax.set_xlabel("δ (rad)")
    ax.set_ylabel("V(δ)")
    ax.grid(True, alpha=0.25)
    plt.show()

    # Plot counts vs wrapped phase after applying best delta
    ph_plot = wrap_to_pi(phi_w + best_d)
    idx = np.argsort(ph_plot)
    cw_best = np.cos(phi_w + best_d)
    lam_fit = np.maximum(best_a + best_b * cw_best, lam_floor)

    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.scatter(ph_plot[idx], yw[idx], s=16, alpha=0.7, label="Observed counts")
    ax.plot(ph_plot[idx], lam_fit[idx], linewidth=2, label=r"Fitted $\lambda=\alpha+\beta\cos(\phi+\delta)$")
    ax.set_title(f"Best fit in phase domain: α={best_a:.3g}, β={best_b:.3g}, V={best_V:.4f}")
    ax.set_xlabel("wrapped phase (rad)")
    ax.set_ylabel("counts / expected counts")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    # Now: show the IRLS iteration trace for the BEST delta
    cw = np.cos(phi_w + best_d)
    a, b, nll, trace = irls_fit_alpha_beta_trace(yw, cw, iters=iters_ab, lam_floor=lam_floor)

    print("\n[IRLS iteration trace at best δ]")
    print("  iter   alpha        beta         V=beta/alpha   NLL          det        lam_min   lam_max")
    for row in trace:
        note = row.get("note", "")
        print(f"  {row['iter']:>4d}  {row['alpha']:>10.6g}  {row['beta']:>10.6g}  {row['V']:>12.6g}  "
              f"{row['nll']:>10.6g}  {row['det']:>9.3e}  {row['lam_min']:>8.3g}  {row['lam_max']:>8.3g} {note}")

    return gs

# -----------------------------
# Example call with your variables
# -----------------------------
ts = DNF[4]['data']['timestamps'][0:301]
counts = DNF[4]['data']['counts'][0:301]
pm_volts = DNF[4]['data']['pm_volts'][0:301]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])
f_ramp = 1000.0
tau = 3e-3
#
gs = show_window_mle_diagnostics(
     ts, counts, pm_volts, V2pi,
     f_ramp=f_ramp, tau=tau, step=tau/2,
     window_index=3,
     n_phase=128,
     iters_ab=10,
    lam_floor=1e-9,
     use_pm_phase=True,   # try True, then False and compare
 )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt
# ============================================================
# Helpers (match your existing logic)
# ============================================================
def to_seconds(ts):
    ts = np.asarray(ts)
    if np.issubdtype(ts.dtype, np.datetime64):
        t0 = ts[0]
        return (ts - t0) / np.timedelta64(1, "s")
    return ts.astype(float) - float(ts[0])

def wrap_to_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def poisson_nll(y, lam, lam_floor=1e-12):
    lam = np.maximum(lam, lam_floor)
    return float(np.sum(lam - y*np.log(lam)))  # +const ignored

def _sigmoid(z):
    # stable sigmoid
    if z >= 0:
        ez = np.exp(-z)
        return 1.0 / (1.0 + ez)
    else:
        ez = np.exp(z)
        return ez / (1.0 + ez)

# ============================================================
# Method 1: IRLS-like solver for (alpha, beta) given cw
# ============================================================
def irls_fit_alpha_beta_trace(yw, cw, iters=8, lam_floor=1e-6):
    """
    Step-by-step IRLS-like solver: lambda = alpha + beta*cw
    Constraints: alpha>=lam_floor, beta>=0, beta<=alpha-lam_floor
    Returns (alpha, beta, nll, trace)
    """
    yw = np.asarray(yw, dtype=float)
    cw = np.asarray(cw, dtype=float)

    alpha = max(float(np.mean(yw)), lam_floor)
    beta = 0.0
    trace = []

    for k in range(int(iters)):
        lam = alpha + beta * cw
        lam = np.maximum(lam, lam_floor)
        w = 1.0 / lam

        sw0  = float(np.sum(w))
        sw1  = float(np.sum(w * cw))
        sw2  = float(np.sum(w * cw * cw))
        swy0 = float(np.sum(w * yw))
        swy1 = float(np.sum(w * yw * cw))

        det = sw0 * sw2 - sw1 * sw1
        if abs(det) < 1e-18:
            trace.append({
                "iter": k+1, "alpha": alpha, "beta": beta, "V": (beta/alpha if alpha>0 else np.nan),
                "nll": poisson_nll(yw, lam, lam_floor), "det": det,
                "lam_min": float(lam.min()), "lam_max": float(lam.max()),
                "note": "det ~ 0, stopped"
            })
            break

        alpha_new = ( sw2 * swy0 - sw1 * swy1 ) / det
        beta_new  = (-sw1 * swy0 + sw0 * swy1 ) / det

        # constraints
        if alpha_new < lam_floor:
            alpha_new = lam_floor
        if beta_new < 0.0:
            beta_new = 0.0
        if beta_new > alpha_new - lam_floor:
            beta_new = max(0.0, alpha_new - lam_floor)

        alpha, beta = float(alpha_new), float(beta_new)
        lam = np.maximum(alpha + beta*cw, lam_floor)
        nll = poisson_nll(yw, lam, lam_floor)
        trace.append({
            "iter": k+1,
            "alpha": alpha,
            "beta": beta,
            "V": (beta/alpha if alpha>0 else np.nan),
            "nll": nll,
            "det": det,
            "lam_min": float(lam.min()),
            "lam_max": float(lam.max()),
        })

    lam = np.maximum(alpha + beta*cw, lam_floor)
    nll = poisson_nll(yw, lam, lam_floor)
    return alpha, beta, nll, trace


def grid_search_delta_irls(yw, phi_w, n_phase=64, iters_ab=8, lam_floor=1e-6):
    """
    Try deltas, fit (alpha,beta) with IRLS, compute Poisson NLL, pick best.
    """
    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False)
    nlls = np.full(deltas.size, np.nan)
    alphas = np.full(deltas.size, np.nan)
    betas = np.full(deltas.size, np.nan)
    Vs = np.full(deltas.size, np.nan)

    for i, d in enumerate(deltas):
        cw = np.cos(phi_w + d)
        a, b, nll, _ = irls_fit_alpha_beta_trace(yw, cw, iters=iters_ab, lam_floor=lam_floor)
        alphas[i] = a
        betas[i] = b
        nlls[i] = nll
        Vs[i] = (b/a) if a > 0 else np.nan

    best_i = int(np.nanargmin(nlls))
    return {
        "deltas": deltas,
        "nlls": nlls,
        "alphas": alphas,
        "betas": betas,
        "Vs": Vs,
        "best_i": best_i,
        "best_delta": float(deltas[best_i]),
        "best_alpha": float(alphas[best_i]),
        "best_beta": float(betas[best_i]),
        "best_V": float(Vs[best_i]),
    }

# ============================================================
# Method 2: Poisson MLE for (alpha, beta) given cw and fixed delta
# ============================================================
def fit_alpha_beta_poisson_mle_given_delta(yw, cw, lam_floor=1e-12, maxiter=200, tol=1e-9):
    """
    True Poisson MLE fit (alpha,beta) for lambda_i = alpha + beta*cw_i
    Constraints via reparam:
      alpha = exp(a) + lam_floor
      beta  = alpha * sigmoid(b)  in [0, alpha)
    """
    from scipy.optimize import minimize

    yw = np.asarray(yw, float)
    cw = np.asarray(cw, float)

    def unpack(u):
        a, b = float(u[0]), float(u[1])
        alpha = np.exp(a) + lam_floor
        beta  = alpha * _sigmoid(b)
        return alpha, beta

    def obj(u):
        alpha, beta = unpack(u)
        lam = alpha + beta * cw
        return poisson_nll(yw, lam, lam_floor=lam_floor)

    ymean = float(np.mean(yw))
    a0 = np.log(max(ymean, lam_floor) - lam_floor + 1e-18)
    frac0 = 0.3
    b0 = np.log(frac0 / (1.0 - frac0))
    u0 = np.array([a0, b0], dtype=float)

    res = minimize(obj, u0, method="L-BFGS-B",
                   options={"maxiter": int(maxiter), "ftol": float(tol)})
    alpha, beta = unpack(res.x)
    return float(alpha), float(beta), float(res.fun)


def grid_search_delta_poisson_mle(yw, phi_w, n_phase=64, lam_floor=1e-12,
                                  maxiter_ab=200, tol=1e-9):
    """
    Try deltas, fit (alpha,beta) by TRUE Poisson MLE per delta, compute NLL, pick best.
    """
    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False)
    nlls = np.full(deltas.size, np.nan)
    alphas = np.full(deltas.size, np.nan)
    betas = np.full(deltas.size, np.nan)
    Vs = np.full(deltas.size, np.nan)

    for i, d in enumerate(deltas):
        cw = np.cos(phi_w + d)
        a, b, nll = fit_alpha_beta_poisson_mle_given_delta(
            yw, cw, lam_floor=lam_floor, maxiter=maxiter_ab, tol=tol
        )
        alphas[i] = a
        betas[i] = b
        nlls[i] = nll
        Vs[i] = (b/a) if a > 0 else np.nan

    best_i = int(np.nanargmin(nlls))
    return {
        "deltas": deltas,
        "nlls": nlls,
        "alphas": alphas,
        "betas": betas,
        "Vs": Vs,
        "best_i": best_i,
        "best_delta": float(deltas[best_i]),
        "best_alpha": float(alphas[best_i]),
        "best_beta": float(betas[best_i]),
        "best_V": float(Vs[best_i]),
    }

# ============================================================
# Method 3: Full joint Poisson MLE over (alpha, beta, delta)
# ============================================================
def fit_alpha_beta_delta_poisson_mle_full(yw, phi_w, lam_floor=1e-12,
                                          delta_init_grid=64, maxiter=300, tol=1e-9):
    """
    Full joint Poisson MLE over (alpha,beta,delta), continuous delta.
    Reparam:
      alpha = exp(a) + lam_floor
      beta  = alpha * sigmoid(b)
      delta free (wrapped for reporting)
    """
    from scipy.optimize import minimize

    yw = np.asarray(yw, float)
    phi_w = np.asarray(phi_w, float)

    def unpack(p):
        a, b, d = float(p[0]), float(p[1]), float(p[2])
        alpha = np.exp(a) + lam_floor
        beta  = alpha * _sigmoid(b)
        delta = wrap_to_pi(d)
        return alpha, beta, delta

    def obj(p):
        alpha, beta, delta = unpack(p)
        lam = alpha + beta * np.cos(phi_w + delta)
        return poisson_nll(yw, lam, lam_floor=lam_floor)

    # start from coarse delta grid
    ymean = float(np.mean(yw))
    a0 = np.log(max(ymean, lam_floor) - lam_floor + 1e-18)
    frac0 = 0.3
    b0 = np.log(frac0 / (1.0 - frac0))

    deltas = np.linspace(-np.pi, np.pi, int(delta_init_grid), endpoint=False)
    alpha0 = max(ymean, lam_floor)
    beta0  = 0.25 * alpha0
    best = (np.inf, 0.0)
    for d in deltas:
        lam = alpha0 + beta0 * np.cos(phi_w + d)
        nll = poisson_nll(yw, lam, lam_floor=lam_floor)
        if nll < best[0]:
            best = (nll, d)
    d0 = float(best[1])

    p0 = np.array([a0, b0, d0], dtype=float)
    res = minimize(obj, p0, method="L-BFGS-B",
                   options={"maxiter": int(maxiter), "ftol": float(tol)})

    alpha, beta, delta = unpack(res.x)
    return float(alpha), float(beta), float(delta), float(res.fun)

# ============================================================
# Plotting helper: per method
# ============================================================
def plot_method_suite(method_name, tw, yw, phi_w, best_delta, best_alpha, best_beta,
                      deltas=None, nlls=None, Vs=None, lam_floor=1e-12,
                      show_grid_plots=True):
    """
    For ONE method, produce the full set of plots requested.
    """
    # 1) Plot windowed counts vs time
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(tw, yw, linewidth=1)
    ax.set_title(f"[{method_name}] Windowed counts vs time")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("counts")
    ax.grid(True, alpha=0.25)
    plt.show()

    # 2) Grid search over delta (plots)
    if show_grid_plots and (deltas is not None) and (nlls is not None) and (Vs is not None):
        # 3) Plot NLL vs delta
        fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
        ax.plot(deltas, nlls, linewidth=1)
        ax.axvline(best_delta, linestyle="--", linewidth=1)
        ax.set_title(f"[{method_name}] Poisson NLL vs δ (best δ={best_delta:+.3f} rad)")
        ax.set_xlabel("δ (rad)")
        ax.set_ylabel("Poisson NLL")
        ax.grid(True, alpha=0.25)
        plt.show()

        # 4) Plot V vs delta
        fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
        ax.plot(deltas, Vs, linewidth=1)
        ax.axvline(best_delta, linestyle="--", linewidth=1)
        ax.set_title(f"[{method_name}] V(δ)=β/α vs δ (best V={(best_beta/best_alpha if best_alpha>0 else np.nan):.4f})")
        ax.set_xlabel("δ (rad)")
        ax.set_ylabel("V(δ)")
        ax.grid(True, alpha=0.25)
        plt.show()

    # 5) Plot counts vs wrapped phase after applying best delta
    ph_plot = wrap_to_pi(phi_w + best_delta)
    idx = np.argsort(ph_plot)
    cw_best = np.cos(phi_w + best_delta)
    lam_fit = np.maximum(best_alpha + best_beta * cw_best, lam_floor)
    V_best = (best_beta / best_alpha) if best_alpha > 0 else np.nan

    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.scatter(ph_plot[idx], yw[idx], s=16, alpha=0.7, label="Observed counts")
    ax.plot(ph_plot[idx], lam_fit[idx], linewidth=2, label=r"Fitted $\lambda=\alpha+\beta\cos(\phi+\delta)$")
    ax.set_title(f"[{method_name}] Best fit in phase domain: α={best_alpha:.4g}, β={best_beta:.4g}, δ={best_delta:+.4f}, V={V_best:.5f}")
    ax.set_xlabel("wrapped phase (rad)")
    ax.set_ylabel("counts / expected counts")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()


# ============================================================
# Main diagnostic: plots the full suite for EACH method
# ============================================================
def show_window_mle_diagnostics_all_methods(
    ts,
    counts,
    pm_volts,
    V2pi,
    f_ramp=1000.0,
    tau=3e-3,
    step=None,
    window_index=0,
    n_phase=64,
    iters_ab=8,
    lam_floor=1e-9,
    use_pm_phase=True,     # True: use pm_volts→phase, False: use ideal ramp from t
    maxiter_mle_ab=250,
    tol_mle_ab=1e-10,
    maxiter_full=400,
    tol_full=1e-10,
):
    """
    Produces, for each method:
      - Plot windowed counts vs time
      - Grid search over delta (implicit in NLL/V plots)
      - Plot NLL vs delta
      - Plot V vs delta
      - Plot counts vs wrapped phase after applying best delta
    """
    t_s = to_seconds(ts).astype(np.float64)
    y = np.asarray(counts, dtype=float)
    pm = np.asarray(pm_volts, dtype=float)

    n = min(t_s.size, y.size, pm.size)
    t_s = t_s[:n]; y = y[:n]; pm = pm[:n]
    if n < 3:
        raise ValueError("Need >=3 samples.")

    dt = float(np.mean(np.diff(t_s)))
    if step is None:
        step = tau/2.0
    K = max(3, int(round(tau/dt)))
    S = max(1, int(round(step/dt)))

    start = window_index * S
    end = start + K
    if end > n:
        raise ValueError("window_index too large for this tau/step/data length.")

    yw = y[start:end]
    tw = t_s[start:end]

    # Phase model
    if use_pm_phase:
        phi = 2*np.pi * (pm / float(V2pi))
        phi = np.unwrap(phi)  # avoid sawtooth wrap artifacts
        phase_label = "PM volts → phase (unwrapped)"
    else:
        phi = 2*np.pi * float(f_ramp) * t_s
        phase_label = "Ideal ramp phase 2π f t"

    phi_w = phi[start:end]

    print(f"[Window diagnostics]")
    print(f"  use_phase_model: {phase_label}")
    print(f"  n_samples total: {n}")
    print(f"  dt ≈ {dt*1e6:.2f} µs  (Fs ≈ {1/dt/1e3:.2f} kHz)")
    print(f"  tau_req={tau:.3e} s -> K={K} samples -> tau_eff={K*dt:.3e} s")
    print(f"  step_req={step:.3e} s -> S={S} samples -> step_eff={S*dt:.3e} s")
    print(f"  window_index={window_index} => idx [{start}:{end})")
    print(f"  sum counts in window: {float(np.sum(yw)):.1f}, mean/bin: {float(np.mean(yw)):.3f}")

    results = {}

    # -----------------------------
    # Method 1: IRLS grid
    # -----------------------------
    gs_irls = grid_search_delta_irls(
        yw, phi_w, n_phase=n_phase, iters_ab=iters_ab, lam_floor=lam_floor
    )
    plot_method_suite(
        "IRLS grid (α,β via IRLS per δ)",
        tw, yw, phi_w,
        best_delta=gs_irls["best_delta"],
        best_alpha=gs_irls["best_alpha"],
        best_beta=gs_irls["best_beta"],
        deltas=gs_irls["deltas"],
        nlls=gs_irls["nlls"],
        Vs=gs_irls["Vs"],
        lam_floor=lam_floor,
        show_grid_plots=True
    )

    # IRLS iteration trace for best delta (print)
    cw_best = np.cos(phi_w + gs_irls["best_delta"])
    a, b, nll, trace = irls_fit_alpha_beta_trace(yw, cw_best, iters=iters_ab, lam_floor=lam_floor)
    print("\n[IRLS iteration trace at best δ (IRLS-grid method)]")
    print("  iter   alpha        beta         V=beta/alpha   NLL          det        lam_min   lam_max")
    for row in trace:
        note = row.get("note", "")
        print(f"  {row['iter']:>4d}  {row['alpha']:>10.6g}  {row['beta']:>10.6g}  {row['V']:>12.6g}  "
              f"{row['nll']:>10.6g}  {row['det']:>9.3e}  {row['lam_min']:>8.3g}  {row['lam_max']:>8.3g} {note}")

    results["irls_grid"] = gs_irls

    # -----------------------------
    # Method 2: Poisson-MLE grid (α,β true MLE per δ)
    # -----------------------------
    gs_mlegrid = grid_search_delta_poisson_mle(
        yw, phi_w, n_phase=n_phase, lam_floor=lam_floor,
        maxiter_ab=maxiter_mle_ab, tol=tol_mle_ab
    )
    plot_method_suite(
        "Poisson-MLE grid (α,β true MLE per δ)",
        tw, yw, phi_w,
        best_delta=gs_mlegrid["best_delta"],
        best_alpha=gs_mlegrid["best_alpha"],
        best_beta=gs_mlegrid["best_beta"],
        deltas=gs_mlegrid["deltas"],
        nlls=gs_mlegrid["nlls"],
        Vs=gs_mlegrid["Vs"],
        lam_floor=lam_floor,
        show_grid_plots=True
    )
    results["poisson_mle_grid"] = gs_mlegrid

    # -----------------------------
    # Method 3: Full continuous Poisson MLE (α,β,δ jointly)
    # For fairness: also compute a "profile" NLL/V vs δ curve using Poisson-MLE grid
    # and just mark the full-MLE point on those curves.
    # -----------------------------
    a_full, b_full, d_full, nll_full = fit_alpha_beta_delta_poisson_mle_full(
        yw, phi_w, lam_floor=lam_floor, delta_init_grid=n_phase,
        maxiter=maxiter_full, tol=tol_full
    )
    V_full = (b_full / a_full) if a_full > 0 else np.nan
    print("\n[Full continuous Poisson MLE]")
    print(f"  alpha={a_full:.6g}, beta={b_full:.6g}, delta={d_full:+.6f} rad, V={V_full:.6f}, NLL={nll_full:.6g}")

    # Profile curves from Poisson-MLE grid (already computed): gs_mlegrid
    # We'll plot NLL(δ) and V(δ) again but highlight the full-MLE delta & V
    d = gs_mlegrid["deltas"]
    nlls = gs_mlegrid["nlls"]
    Vs = gs_mlegrid["Vs"]

    # Plot windowed counts vs time (again, per your request "for each method")
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(tw, yw, linewidth=1)
    ax.set_title("[Full continuous Poisson MLE] Windowed counts vs time")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("counts")
    ax.grid(True, alpha=0.25)
    plt.show()

    # Plot NLL vs delta (profile) with full-MLE point marked
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(d, nlls, linewidth=1, label="profile NLL(δ): Poisson-MLE over (α,β) for each δ")
    ax.axvline(d_full, linestyle="--", linewidth=1, label=f"full MLE δ={d_full:+.3f}")
    ax.set_title("[Full continuous Poisson MLE] Profile Poisson NLL vs δ")
    ax.set_xlabel("δ (rad)")
    ax.set_ylabel("Poisson NLL")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    # Plot V vs delta (profile) with full-MLE V marked
    fig, ax = plt.subplots(figsize=(9, 3.0), constrained_layout=True)
    ax.plot(d, Vs, linewidth=1, label="profile V(δ): Poisson-MLE over (α,β) for each δ")
    ax.axvline(d_full, linestyle="--", linewidth=1, label=f"full MLE δ={d_full:+.3f}")
    ax.axhline(V_full, linestyle=":", linewidth=1, label=f"full MLE V={V_full:.4f}")
    ax.set_title("[Full continuous Poisson MLE] Profile V(δ) vs δ")
    ax.set_xlabel("δ (rad)")
    ax.set_ylabel("V(δ)")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    # Plot counts vs wrapped phase using full-MLE delta
    plot_method_suite(
        "Full continuous Poisson MLE (fit α,β,δ jointly)",
        tw, yw, phi_w,
        best_delta=d_full,
        best_alpha=a_full,
        best_beta=b_full,
        # no intrinsic grid for full MLE; we already plotted "grid/profile" above
        deltas=None, nlls=None, Vs=None,
        lam_floor=lam_floor,
        show_grid_plots=False
    )

    results["full_mle"] = {
        "alpha": a_full,
        "beta": b_full,
        "delta": d_full,
        "V": V_full,
        "nll": nll_full,
        "profile_from_mle_grid": gs_mlegrid,  # reuse for profiling curves
    }

    return results


# ============================================================
# Example call with your variables
# ============================================================
ts = DNF[4]['data']['timestamps'][0:301]
counts = DNF[4]['data']['counts'][0:301]
pm_volts = DNF[4]['data']['pm_volts'][0:301]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])
f_ramp = 1000.0
tau = 3e-3

results = show_window_mle_diagnostics_all_methods(
     ts, counts, pm_volts, V2pi,
     f_ramp=f_ramp, tau=tau, step=tau/2,
     window_index=3,
     n_phase=256,
     iters_ab=8,
     lam_floor=1e-6,
     use_pm_phase=True,  # try True, then False and compare
)


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Assumes these already exist in scope (from your prior script):
#   to_seconds, wrap_to_pi, poisson_nll, _sigmoid
#   grid_search_delta_irls, grid_search_delta_poisson_mle
#   fit_alpha_beta_delta_poisson_mle_full
# ============================================================

def _ensure_seconds(ts):
    return to_seconds(ts).astype(np.float64)

def _compute_phase(ts_s, pm_volts, V2pi, f_ramp, phase_offset=0.0, use_pm_phase=True):
    if use_pm_phase:
        phi = 2*np.pi * (np.asarray(pm_volts, float) / float(V2pi)) + float(phase_offset)
        phi = np.unwrap(phi)
        return phi
    else:
        return (2*np.pi * float(f_ramp) * np.asarray(ts_s, float) + float(phase_offset))

def _window_indices(n, K, S):
    if K > n:
        return np.array([], dtype=int), np.array([], dtype=int)
    nwin = (n - K) // S + 1
    starts = (np.arange(nwin, dtype=int) * S)
    ends = starts + K
    return starts, ends

def compare_window_params_vs_full_mle_timed(
    ts,
    counts,
    pm_volts,
    V2pi,
    f_ramp: float,
    taus,
    overlap: float = 0.5,
    use_pm_phase: bool = True,
    phase_offset: float = 0.0,
    n_min: float = 0.0,
    lam_floor: float = 1e-9,
    # sweep knobs:
    n_phase_list=(64,),
    iters_ab_list=(8,),
    # method 2 options:
    maxiter_mle_ab: int = 250,
    tol_mle_ab: float = 1e-10,
    # full continuous options:
    delta_init_grid: int = 64,
    maxiter_full: int = 300,
    tol_full: float = 1e-10,
    # runtime controls:
    max_windows_per_tau: int = 300,
    window_subsample: int = 1,
    # output:
    make_plots: bool = True,
    print_timing: bool = True,
):
    """
    Same as compare_window_params_vs_full_mle, but also TIMES each method.

    Timing is recorded per (tau, n_phase, iters_ab):
      - total wall time for each method across all evaluated windows
      - average time per evaluated window
    """
    ts_s = _ensure_seconds(ts)
    y = np.asarray(counts, float)
    pm = np.asarray(pm_volts, float)
    n = int(min(ts_s.size, y.size, pm.size))
    ts_s = ts_s[:n]; y = y[:n]; pm = pm[:n]

    if n < 5:
        raise ValueError("Need more samples.")

    phi = _compute_phase(ts_s, pm, V2pi, f_ramp, phase_offset=phase_offset, use_pm_phase=use_pm_phase)

    dts = np.diff(ts_s)
    dt_pos = dts[dts > 0]
    if dt_pos.size == 0:
        raise ValueError("Timestamps not strictly increasing.")
    dt = float(np.mean(dt_pos))

    taus = np.asarray(taus, float)
    taus = taus[np.isfinite(taus)]
    taus = np.unique(taus)
    if taus.size == 0:
        raise ValueError("No valid taus supplied.")
    if not (0.0 <= overlap < 1.0):
        raise ValueError("overlap must be in [0,1).")

    out = {
        "meta": {
            "dt": dt,
            "n_samples": n,
            "f_ramp": float(f_ramp),
            "overlap": float(overlap),
            "use_pm_phase": bool(use_pm_phase),
            "phase_offset": float(phase_offset),
            "lam_floor": float(lam_floor),
            "n_min": float(n_min),
        },
        "results": {}  # key: (tau, n_phase, iters_ab)
    }

    # ------------------------------------------------------------
    # main sweep
    # ------------------------------------------------------------
    for tau in taus:
        K = max(3, int(round(float(tau) / dt)))
        step = float(tau) * (1.0 - overlap)
        S = max(1, int(round(step / dt)))

        starts, ends = _window_indices(n, K, S)
        if starts.size == 0:
            continue

        sel = np.arange(starts.size)[::max(1, int(window_subsample))]
        if sel.size > max_windows_per_tau:
            sel = sel[:max_windows_per_tau]
        starts = starts[sel]
        ends = ends[sel]

        for n_phase in n_phase_list:
            for iters_ab in iters_ab_list:
                key = (float(tau), int(n_phase), int(iters_ab))

                nwin = starts.size
                alpha_full = np.full(nwin, np.nan)
                beta_full  = np.full(nwin, np.nan)
                delta_full = np.full(nwin, np.nan)

                alpha_irls = np.full(nwin, np.nan)
                beta_irls  = np.full(nwin, np.nan)
                delta_irls = np.full(nwin, np.nan)

                alpha_mlegrid = np.full(nwin, np.nan)
                beta_mlegrid  = np.full(nwin, np.nan)
                delta_mlegrid = np.full(nwin, np.nan)

                # ---- Timing accumulators
                t_full_total = 0.0
                t_irls_total = 0.0
                t_mlegrid_total = 0.0

                n_eval_full = 0
                n_eval_irls = 0
                n_eval_mlegrid = 0

                for w in range(nwin):
                    s = int(starts[w]); e = int(ends[w])
                    yw = y[s:e]
                    sy = float(np.sum(yw))
                    if sy < float(n_min):
                        continue
                    phi_w = phi[s:e]

                    # --- FULL continuous MLE
                    t0 = time.perf_counter()
                    aF, bF, dF, nllF = fit_alpha_beta_delta_poisson_mle_full(
                        yw, phi_w, lam_floor=lam_floor,
                        delta_init_grid=int(delta_init_grid),
                        maxiter=int(maxiter_full),
                        tol=float(tol_full),
                    )
                    t_full_total += (time.perf_counter() - t0)
                    n_eval_full += 1
                    alpha_full[w] = aF
                    beta_full[w]  = bF
                    delta_full[w] = dF

                    # --- IRLS grid
                    t0 = time.perf_counter()
                    gsI = grid_search_delta_irls(
                        yw, phi_w, n_phase=int(n_phase),
                        iters_ab=int(iters_ab), lam_floor=lam_floor
                    )
                    t_irls_total += (time.perf_counter() - t0)
                    n_eval_irls += 1
                    alpha_irls[w] = gsI["best_alpha"]
                    beta_irls[w]  = gsI["best_beta"]
                    delta_irls[w] = gsI["best_delta"]

                    # --- Poisson-MLE grid
                    t0 = time.perf_counter()
                    gsM = grid_search_delta_poisson_mle(
                        yw, phi_w, n_phase=int(n_phase),
                        lam_floor=lam_floor, maxiter_ab=int(maxiter_mle_ab), tol=float(tol_mle_ab)
                    )
                    t_mlegrid_total += (time.perf_counter() - t0)
                    n_eval_mlegrid += 1
                    alpha_mlegrid[w] = gsM["best_alpha"]
                    beta_mlegrid[w]  = gsM["best_beta"]
                    delta_mlegrid[w] = gsM["best_delta"]

                # differences vs full
                d_alpha_irls = alpha_irls - alpha_full
                d_beta_irls  = beta_irls  - beta_full
                d_delta_irls = wrap_to_pi(delta_irls - delta_full)

                d_alpha_mlegrid = alpha_mlegrid - alpha_full
                d_beta_mlegrid  = beta_mlegrid  - beta_full
                d_delta_mlegrid = wrap_to_pi(delta_mlegrid - delta_full)

                # summary stats
                def _summ(x):
                    m = np.isfinite(x)
                    if not np.any(m):
                        return {"n": 0}
                    xm = x[m]
                    return {
                        "n": int(xm.size),
                        "mean": float(np.mean(xm)),
                        "median": float(np.median(xm)),
                        "p10": float(np.quantile(xm, 0.10)),
                        "p90": float(np.quantile(xm, 0.90)),
                        "rms": float(np.sqrt(np.mean(xm*xm))),
                    }

                timing = {
                    "full_mle": {
                        "total_s": float(t_full_total),
                        "n_eval": int(n_eval_full),
                        "avg_s_per_window": float(t_full_total / n_eval_full) if n_eval_full else np.nan,
                    },
                    "irls_grid": {
                        "total_s": float(t_irls_total),
                        "n_eval": int(n_eval_irls),
                        "avg_s_per_window": float(t_irls_total / n_eval_irls) if n_eval_irls else np.nan,
                    },
                    "mle_grid": {
                        "total_s": float(t_mlegrid_total),
                        "n_eval": int(n_eval_mlegrid),
                        "avg_s_per_window": float(t_mlegrid_total / n_eval_mlegrid) if n_eval_mlegrid else np.nan,
                    }
                }

                out["results"][key] = {
                    "tau": float(tau),
                    "K": int(K),
                    "S": int(S),
                    "n_windows_used": int(np.sum(np.isfinite(alpha_full))),
                    "timing": timing,
                    "arrays": {
                        "alpha_full": alpha_full, "beta_full": beta_full, "delta_full": delta_full,
                        "alpha_irls": alpha_irls, "beta_irls": beta_irls, "delta_irls": delta_irls,
                        "alpha_mlegrid": alpha_mlegrid, "beta_mlegrid": beta_mlegrid, "delta_mlegrid": delta_mlegrid,
                        "d_alpha_irls": d_alpha_irls, "d_beta_irls": d_beta_irls, "d_delta_irls": d_delta_irls,
                        "d_alpha_mlegrid": d_alpha_mlegrid, "d_beta_mlegrid": d_beta_mlegrid, "d_delta_mlegrid": d_delta_mlegrid,
                    },
                    "summary": {
                        "irls_minus_full": {
                            "d_alpha": _summ(d_alpha_irls),
                            "d_beta":  _summ(d_beta_irls),
                            "d_delta": _summ(d_delta_irls),
                        },
                        "mlegrid_minus_full": {
                            "d_alpha": _summ(d_alpha_mlegrid),
                            "d_beta":  _summ(d_beta_mlegrid),
                            "d_delta": _summ(d_delta_mlegrid),
                        },
                    }
                }

                if print_timing:
                    tinfo = out["results"][key]["timing"]
                    print(f"\n[tau={tau:.3e}s | n_phase={n_phase} | iters_ab={iters_ab} | windows={nwin}]")
                    print(f"  FULL MLE:    total={tinfo['full_mle']['total_s']:.3f}s  "
                          f"n={tinfo['full_mle']['n_eval']}  avg={tinfo['full_mle']['avg_s_per_window']*1e3:.3f} ms/win")
                    print(f"  IRLS grid:   total={tinfo['irls_grid']['total_s']:.3f}s  "
                          f"n={tinfo['irls_grid']['n_eval']}  avg={tinfo['irls_grid']['avg_s_per_window']*1e3:.3f} ms/win")
                    print(f"  MLE grid:    total={tinfo['mle_grid']['total_s']:.3f}s  "
                          f"n={tinfo['mle_grid']['n_eval']}  avg={tinfo['mle_grid']['avg_s_per_window']*1e3:.3f} ms/win")

                # Optional per-config distribution plots (same as before)
                if make_plots:
                    fig, axs = plt.subplots(3, 2, figsize=(10, 8), constrained_layout=True)
                    fig.suptitle(
                        f"Differences vs FULL MLE | tau={tau:.3e}s (K={K},S={S}) | n_phase={n_phase}, iters_ab={iters_ab}\n"
                        f"Timing avg (ms/win): full={timing['full_mle']['avg_s_per_window']*1e3:.2f}, "
                        f"IRLS={timing['irls_grid']['avg_s_per_window']*1e3:.2f}, "
                        f"MLEgrid={timing['mle_grid']['avg_s_per_window']*1e3:.2f}",
                        fontsize=10
                    )

                    # IRLS - FULL
                    axs[0,0].hist(d_alpha_irls[np.isfinite(d_alpha_irls)], bins=50)
                    axs[0,0].set_title("IRLS−FULL: Δalpha"); axs[0,0].grid(True, alpha=0.2)

                    axs[1,0].hist(d_beta_irls[np.isfinite(d_beta_irls)], bins=50)
                    axs[1,0].set_title("IRLS−FULL: Δbeta"); axs[1,0].grid(True, alpha=0.2)

                    axs[2,0].hist(d_delta_irls[np.isfinite(d_delta_irls)], bins=50)
                    axs[2,0].set_title("IRLS−FULL: Δdelta (wrapped)"); axs[2,0].grid(True, alpha=0.2)

                    # MLEgrid - FULL
                    axs[0,1].hist(d_alpha_mlegrid[np.isfinite(d_alpha_mlegrid)], bins=50)
                    axs[0,1].set_title("MLEgrid−FULL: Δalpha"); axs[0,1].grid(True, alpha=0.2)

                    axs[1,1].hist(d_beta_mlegrid[np.isfinite(d_beta_mlegrid)], bins=50)
                    axs[1,1].set_title("MLEgrid−FULL: Δbeta"); axs[1,1].grid(True, alpha=0.2)

                    axs[2,1].hist(d_delta_mlegrid[np.isfinite(d_delta_mlegrid)], bins=50)
                    axs[2,1].set_title("MLEgrid−FULL: Δdelta (wrapped)"); axs[2,1].grid(True, alpha=0.2)

                    for r in range(3):
                        for c in range(2):
                            axs[r,c].set_ylabel("count")
                    plt.show()

    # ------------------------------------------------------------
    # Summary plot across taus with timing overlays
    # ------------------------------------------------------------
    if make_plots and len(out["results"]) > 0:
        keys = list(out["results"].keys())
        taus_u = sorted(set([k[0] for k in keys]))
        cfgs = sorted(set([(k[1], k[2]) for k in keys]))  # (n_phase, iters)

        for (n_phase, iters_ab) in cfgs:
            taus_arr = np.asarray(taus_u, float)

            # Collect RMS diffs and timing
            rms_a_irls, rms_b_irls, rms_d_irls = [], [], []
            rms_a_mle,  rms_b_mle,  rms_d_mle  = [], [], []
            t_full_ms, t_irls_ms, t_mle_ms = [], [], []

            for tau in taus_u:
                r = out["results"].get((float(tau), int(n_phase), int(iters_ab)))
                if r is None:
                    rms_a_irls.append(np.nan); rms_b_irls.append(np.nan); rms_d_irls.append(np.nan)
                    rms_a_mle.append(np.nan);  rms_b_mle.append(np.nan);  rms_d_mle.append(np.nan)
                    t_full_ms.append(np.nan); t_irls_ms.append(np.nan); t_mle_ms.append(np.nan)
                    continue

                s1 = r["summary"]["irls_minus_full"]
                s2 = r["summary"]["mlegrid_minus_full"]
                rms_a_irls.append(s1["d_alpha"].get("rms", np.nan))
                rms_b_irls.append(s1["d_beta"].get("rms", np.nan))
                rms_d_irls.append(s1["d_delta"].get("rms", np.nan))

                rms_a_mle.append(s2["d_alpha"].get("rms", np.nan))
                rms_b_mle.append(s2["d_beta"].get("rms", np.nan))
                rms_d_mle.append(s2["d_delta"].get("rms", np.nan))

                tinfo = r["timing"]
                t_full_ms.append(tinfo["full_mle"]["avg_s_per_window"] * 1e3)
                t_irls_ms.append(tinfo["irls_grid"]["avg_s_per_window"] * 1e3)
                t_mle_ms.append(tinfo["mle_grid"]["avg_s_per_window"] * 1e3)

            fig, axs = plt.subplots(2, 3, figsize=(12.5, 6.5), constrained_layout=True)
            fig.suptitle(f"Accuracy (RMS vs FULL) + Timing | n_phase={n_phase}, iters_ab={iters_ab}", fontsize=11)

            # Row 1: RMS diffs
            axs[0,0].plot(taus_arr, rms_a_irls, marker="o", label="IRLS−FULL")
            axs[0,0].plot(taus_arr, rms_a_mle,  marker="o", label="MLEgrid−FULL")
            axs[0,0].set_xscale("log"); axs[0,0].set_title("RMS(Δalpha)")
            axs[0,0].set_xlabel("tau (s)"); axs[0,0].grid(True, alpha=0.2); axs[0,0].legend()

            axs[0,1].plot(taus_arr, rms_b_irls, marker="o", label="IRLS−FULL")
            axs[0,1].plot(taus_arr, rms_b_mle,  marker="o", label="MLEgrid−FULL")
            axs[0,1].set_xscale("log"); axs[0,1].set_title("RMS(Δbeta)")
            axs[0,1].set_xlabel("tau (s)"); axs[0,1].grid(True, alpha=0.2)

            axs[0,2].plot(taus_arr, rms_d_irls, marker="o", label="IRLS−FULL")
            axs[0,2].plot(taus_arr, rms_d_mle,  marker="o", label="MLEgrid−FULL")
            axs[0,2].set_xscale("log"); axs[0,2].set_title("RMS(Δdelta) [rad]")
            axs[0,2].set_xlabel("tau (s)"); axs[0,2].grid(True, alpha=0.2)

            # Row 2: timing per window
            axs[1,0].plot(taus_arr, t_full_ms, marker="o", label="FULL MLE")
            axs[1,0].set_xscale("log"); axs[1,0].set_title("Avg time/window (ms): FULL")
            axs[1,0].set_xlabel("tau (s)"); axs[1,0].grid(True, alpha=0.2)

            axs[1,1].plot(taus_arr, t_irls_ms, marker="o", label="IRLS grid")
            axs[1,1].set_xscale("log"); axs[1,1].set_title("Avg time/window (ms): IRLS grid")
            axs[1,1].set_xlabel("tau (s)"); axs[1,1].grid(True, alpha=0.2)

            axs[1,2].plot(taus_arr, t_mle_ms, marker="o", label="MLE grid")
            axs[1,2].set_xscale("log"); axs[1,2].set_title("Avg time/window (ms): MLE grid")
            axs[1,2].set_xlabel("tau (s)"); axs[1,2].grid(True, alpha=0.2)

            plt.show()

    return out


# ============================================================
# Example usage
# ============================================================
taus = np.array([ 1e-3, 3e-3, 5e-3])
ts = DNF[4]['data']['timestamps'][0:29999]
counts = DNF[4]['data']['counts'][0:29999]
pm_volts = DNF[4]['data']['pm_volts'][0:29999]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])
out = compare_window_params_vs_full_mle_timed(
    ts, counts, pm_volts, V2pi,
    f_ramp=1000.0,
    taus=taus,
    overlap=0.5,
    use_pm_phase=True,
    n_phase_list=(64, 128, 256),
    iters_ab_list=(8, 12),
    max_windows_per_tau=200,
    window_subsample=2,
    make_plots=False,
    print_timing=True,
)


In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Dict, Any

try:
    from numba import njit
    NUMBA_OK = True
except Exception:
    NUMBA_OK = False

# -----------------------------
# Plot helper (your style)
# -----------------------------
def _plot_with_side_hist(x, y, xlabel, ylabel, ylim, bins=50, title=None):
    from matplotlib.gridspec import GridSpec
    fig = plt.figure(figsize=(10, 4))
    gs = GridSpec(1, 2, width_ratios=[4, 1], wspace=0.10)  # slightly bigger gap

    ax = fig.add_subplot(gs[0, 0])
    ax_hist = fig.add_subplot(gs[0, 1], sharey=ax)

    ax.plot(x, y, lw=1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.25)
    if title:
        ax.set_title(title)

    finite = np.isfinite(y)
    ax_hist.hist(y[finite], bins=bins, orientation="horizontal", density=True, alpha=0.7)
    ax_hist.grid(True, alpha=0.25)
    ax_hist.set_xlabel("PDF")

    plt.setp(ax_hist.get_yticklabels(), visible=False)
    ax_hist.tick_params(axis="y", length=0)
    fig.set_constrained_layout(True)
    plt.show()


# -----------------------------
# Time / phase helpers
# -----------------------------
def _to_seconds(ts: np.ndarray) -> np.ndarray:
    ts = np.asarray(ts)
    if np.issubdtype(ts.dtype, np.datetime64):
        t0 = ts[0]
        return (ts - t0) / np.timedelta64(1, "s")
    return ts.astype(float) - float(ts[0])

def _phase_from_ramp(t_s: np.ndarray, f_ramp: float, phase_offset: float = 0.0) -> np.ndarray:
    return 2.0 * np.pi * f_ramp * t_s + phase_offset

def _phase_from_pm_volts(pm_volts: np.ndarray, V2pi_volts: float, phase_offset: float = 0.0) -> np.ndarray:
    # maps volts -> phase; unwrap for continuity
    phi = 2.0 * np.pi * (pm_volts / float(V2pi_volts)) + float(phase_offset)
    return np.unwrap(phi)

def _wrap_to_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi


# ============================================================
# METHOD 1: YOUR CURRENT IMPLEMENTATION (phase grid + IRLS-like 2x2 solve)
# ============================================================
if not NUMBA_OK:
    raise ImportError("This script needs numba because your current method is numba-jitted.")

@njit(cache=True)
def _poisson_nll(y, lam, lam_floor=1e-12):
    nll = 0.0
    for i in range(y.size):
        li = lam[i]
        if li < lam_floor:
            li = lam_floor
        nll += li - y[i] * np.log(li)
    return nll

@njit(cache=True)
def _fit_alpha_beta_for_delta_irls(yw, cw, iters=6, lam_floor=1e-6):
    # initial
    ymean = 0.0
    for i in range(yw.size):
        ymean += yw[i]
    ymean /= yw.size

    alpha = ymean
    if alpha < lam_floor:
        alpha = lam_floor
    beta = 0.0

    for _ in range(iters):
        sw0 = 0.0; sw1 = 0.0; sw2 = 0.0
        swy0 = 0.0; swy1 = 0.0

        for i in range(yw.size):
            lam = alpha + beta * cw[i]
            if lam < lam_floor:
                lam = lam_floor
            w = 1.0 / lam

            x1 = cw[i]
            sw0 += w
            sw1 += w * x1
            sw2 += w * x1 * x1

            yi = yw[i]
            swy0 += w * yi
            swy1 += w * yi * x1

        det = sw0 * sw2 - sw1 * sw1
        if abs(det) < 1e-18:
            break

        alpha_new = ( sw2 * swy0 - sw1 * swy1) / det
        beta_new  = (-sw1 * swy0 + sw0 * swy1) / det

        if alpha_new < lam_floor:
            alpha_new = lam_floor
        if beta_new < 0.0:
            beta_new = 0.0
        # keep alpha >= beta to avoid negative lambda
        if beta_new > alpha_new - lam_floor:
            beta_new = alpha_new - lam_floor
            if beta_new < 0.0:
                beta_new = 0.0

        alpha = alpha_new
        beta = beta_new

    # final NLL
    lam = np.empty(yw.size, dtype=np.float64)
    for i in range(yw.size):
        li = alpha + beta * cw[i]
        if li < lam_floor:
            li = lam_floor
        lam[i] = li

    nll = _poisson_nll(yw, lam, lam_floor=lam_floor)
    return alpha, beta, nll

@njit(cache=True)
def _mle_phasegrid_params_irls(
    y, cphi, sphi, K, S, n_min, deltas, iters_ab=6, lam_floor=1e-6
):
    """
    Returns per window: alpha,beta,delta,nll,V
    using YOUR current grid+IRLS-like solver.
    """
    n = y.size
    nwin = (n - K) // S + 1

    alpha_out = np.empty(nwin, dtype=np.float64)
    beta_out  = np.empty(nwin, dtype=np.float64)
    delta_out = np.empty(nwin, dtype=np.float64)
    nll_out   = np.empty(nwin, dtype=np.float64)
    V_out     = np.empty(nwin, dtype=np.float64)

    for w in range(nwin):
        start = w * S
        end = start + K

        # count gate
        sy = 0.0
        for i in range(start, end):
            sy += y[i]
        if sy < n_min:
            alpha_out[w] = np.nan
            beta_out[w]  = np.nan
            delta_out[w] = np.nan
            nll_out[w]   = np.nan
            V_out[w]     = np.nan
            continue

        best_nll = 1e300
        best_a = np.nan
        best_b = np.nan
        best_d = 0.0

        for di in range(deltas.size):
            d = deltas[di]
            cd = np.cos(d)
            sd = np.sin(d)

            # build cw and yw
            cw = np.empty(K, dtype=np.float64)
            yw = np.empty(K, dtype=np.float64)
            idx = 0
            for i in range(start, end):
                cw[idx] = cphi[i] * cd - sphi[i] * sd
                yw[idx] = y[i]
                idx += 1

            a, b, nll = _fit_alpha_beta_for_delta_irls(yw, cw, iters=iters_ab, lam_floor=lam_floor)

            if nll < best_nll:
                best_nll = nll
                best_a = a
                best_b = b
                best_d = d

        alpha_out[w] = best_a
        beta_out[w]  = best_b
        delta_out[w] = best_d
        nll_out[w]   = best_nll
        V_out[w]     = best_b / best_a if best_a > 0 else np.nan

    return alpha_out, beta_out, delta_out, nll_out, V_out


# ============================================================
# METHOD 2: FULL CONTINUOUS POISSON MLE (δ optimized continuously)
# ============================================================

def _poisson_nll_py(y, lam, lam_floor=1e-12):
    lam = np.maximum(lam, lam_floor)
    return float(np.sum(lam - y*np.log(lam)))  # const ignored

def _solve_alpha_beta_poisson_newton(y, c, alpha0, beta0, lam_floor=1e-9, maxiter=200, tol=1e-10):
    """
    Proper Poisson MLE for alpha,beta given fixed c=cos(phi+delta):
      maximize sum( y log(alpha+beta c) - (alpha+beta c) )
    Newton steps with damping; constraint alpha>lam_floor, beta>=0, alpha-beta>=lam_floor.
    """
    alpha = float(max(alpha0, lam_floor))
    beta  = float(max(beta0, 0.0))

    for _ in range(int(maxiter)):
        lam = alpha + beta*c
        lam = np.maximum(lam, lam_floor)

        # gradient
        r = y/lam - 1.0
        g0 = np.sum(r)          # d/dalpha
        g1 = np.sum(r * c)      # d/dbeta

        # Hessian (negative definite)
        w = y/(lam*lam)
        H00 = -np.sum(w)
        H01 = -np.sum(w * c)
        H11 = -np.sum(w * c * c)

        det = H00*H11 - H01*H01
        if abs(det) < 1e-18:
            break

        # Newton step: [da, db] = -H^{-1} g
        da = -( H11*g0 - H01*g1) / det
        db = -(-H01*g0 + H00*g1) / det

        # damping line search
        t = 1.0
        nll0 = _poisson_nll_py(y, alpha + beta*c, lam_floor)
        for _ls in range(20):
            a2 = alpha + t*da
            b2 = beta  + t*db

            if a2 < lam_floor:
                a2 = lam_floor
            if b2 < 0.0:
                b2 = 0.0
            if b2 > a2 - lam_floor:
                b2 = max(0.0, a2 - lam_floor)

            nll2 = _poisson_nll_py(y, a2 + b2*c, lam_floor)
            if nll2 <= nll0:
                alpha, beta = a2, b2
                break
            t *= 0.5

        # convergence check on step size
        if abs(t*da) + abs(t*db) < tol*(1.0 + abs(alpha) + abs(beta)):
            break

    return alpha, beta, _poisson_nll_py(y, alpha + beta*c, lam_floor)

def fit_alpha_beta_delta_poisson_mle_full(
    y, phi,
    lam_floor=1e-9,
    delta_init_grid=64,
    maxiter=200,
    tol=1e-10,
):
    """
    Full MLE:
      - coarse grid for delta to get a good start
      - local 1D refinement on delta using golden-section search
      - for each delta evaluation, solve (alpha,beta) by proper Poisson Newton
    Returns alpha,beta,delta,nll
    """
    y = np.asarray(y, float)
    phi = np.asarray(phi, float)

    # coarse grid
    deltas = np.linspace(-np.pi, np.pi, int(delta_init_grid), endpoint=False)
    best = (np.inf, 0.0, 0.0, 0.0)  # (nll, delta, alpha, beta)

    ymean = float(np.mean(y))
    for d in deltas:
        c = np.cos(phi + d)
        a, b, nll = _solve_alpha_beta_poisson_newton(
            y, c, alpha0=ymean, beta0=0.0, lam_floor=lam_floor, maxiter=maxiter, tol=tol
        )
        if nll < best[0]:
            best = (nll, float(d), float(a), float(b))

    # local refine delta with golden-section
    nll_best, d0, a0, b0 = best

    def obj(delta):
        c = np.cos(phi + delta)
        a, b, nll = _solve_alpha_beta_poisson_newton(
            y, c, alpha0=a0, beta0=b0, lam_floor=lam_floor, maxiter=maxiter, tol=tol
        )
        return nll, a, b

    # bracket around d0
    left = d0 - (2*np.pi / float(delta_init_grid))
    right = d0 + (2*np.pi / float(delta_init_grid))

    # golden-section
    gr = (np.sqrt(5) - 1) / 2
    aL = left
    bR = right
    c1 = bR - gr*(bR - aL)
    c2 = aL + gr*(bR - aL)

    f1, a1, b1 = obj(c1)
    f2, a2, b2 = obj(c2)

    for _ in range(40):  # fixed iterations
        if f1 > f2:
            aL = c1
            c1 = c2
            f1, a1, b1 = f2, a2, b2
            c2 = aL + gr*(bR - aL)
            f2, a2, b2 = obj(c2)
        else:
            bR = c2
            c2 = c1
            f2, a2, b2 = f1, a1, b1
            c1 = bR - gr*(bR - aL)
            f1, a1, b1 = obj(c1)

        if abs(bR - aL) < 1e-6:
            break

    if f1 < f2:
        return float(a1), float(b1), float(c1), float(f1)
    else:
        return float(a2), float(b2), float(c2), float(f2)


# ============================================================
# Two-method comparison runner (timed)
# ============================================================
def compare_two_methods_over_time(
    ts,
    counts,
    f_ramp: float,
    tau: float,
    step: Optional[float] = None,
    phase_offset: float = 0.0,
    n_min: float = 0.0,
    clip: bool = True,
    # method 1 (yours):
    n_phase: int = 64,
    iters_ab: int = 8,
    lam_floor: float = 1e-6,
    # method 2 (full):
    delta_init_grid: int = 64,
    full_maxiter: int = 200,
    full_tol: float = 1e-10,
    # phase source:
    pm_volts: Optional[np.ndarray] = None,
    V2pi_volts: Optional[float] = None,
    # plots:
    make_plots: bool = True,
) -> Dict[str, Any]:

    t_s = _to_seconds(np.asarray(ts)).astype(np.float64)
    y = np.clip(np.nan_to_num(np.asarray(counts, float), nan=0.0), 0, None).astype(np.float64)
    n = min(t_s.size, y.size)
    t_s = t_s[:n]; y = y[:n]
    if n < 3:
        raise ValueError("Need at least 3 samples.")

    dts = np.diff(t_s)
    dt_pos = dts[dts > 0]
    if dt_pos.size == 0:
        raise ValueError("Timestamps not strictly increasing.")
    dt = float(np.mean(dt_pos))

    if step is None:
        step = tau/2.0
    K = max(3, int(round(float(tau)/dt)))
    S = max(1, int(round(float(step)/dt)))
    if K > n:
        raise ValueError("K > n for requested tau.")

    # phase model
    using_pm = (pm_volts is not None)
    if using_pm:
        if V2pi_volts is None:
            raise ValueError("Provide V2pi_volts with pm_volts.")
        pv = np.asarray(pm_volts, float)[:n]
        phi = _phase_from_pm_volts(pv, float(V2pi_volts), phase_offset=phase_offset).astype(np.float64)
    else:
        phi = _phase_from_ramp(t_s, f_ramp=f_ramp, phase_offset=phase_offset).astype(np.float64)

    cphi = np.cos(phi).astype(np.float64)
    sphi = np.sin(phi).astype(np.float64)

    nwin = (n - K)//S + 1
    centers_idx = (np.arange(nwin, dtype=np.int64) * S) + (K//2)
    t_centers = t_s[centers_idx]

    # ------------------
    # Method 1 (yours): numba grid+IRLS
    # ------------------
    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False).astype(np.float64)

    t0 = time.perf_counter()
    a1, b1, d1, nll1, V1 = _mle_phasegrid_params_irls(
        y, cphi, sphi, K, S, float(n_min), deltas,
        iters_ab=int(iters_ab), lam_floor=float(lam_floor)
    )
    t_irls = time.perf_counter() - t0

    # ------------------
    # Method 2 (full): python per-window full MLE
    # ------------------
    a2 = np.full(nwin, np.nan)
    b2 = np.full(nwin, np.nan)
    d2 = np.full(nwin, np.nan)
    nll2 = np.full(nwin, np.nan)
    V2 = np.full(nwin, np.nan)

    t0 = time.perf_counter()
    for w in range(nwin):
        s = w*S
        e = s + K
        yw = y[s:e]
        if float(np.sum(yw)) < float(n_min):
            continue
        phi_w = phi[s:e]
        aa, bb, dd, nn = fit_alpha_beta_delta_poisson_mle_full(
            yw, phi_w,
            lam_floor=float(lam_floor),
            delta_init_grid=int(delta_init_grid),
            maxiter=int(full_maxiter),
            tol=float(full_tol)
        )
        a2[w] = aa
        b2[w] = bb
        d2[w] = dd
        nll2[w] = nn
        V2[w] = (bb/aa) if aa > 0 else np.nan
    t_full = time.perf_counter() - t0

    # QBER mapping
    V1q = np.clip(V1, 0, 1) if clip else V1
    Q1 = 0.5*(1.0 - V1q)
    V2q = np.clip(V2, 0, 1) if clip else V2
    Q2 = 0.5*(1.0 - V2q)

    res = {
        "t_centers": t_centers,
        "method_current": {"alpha": a1, "beta": b1, "delta": d1, "nll": nll1, "V": V1, "Q": Q1, "time_s": t_irls},
        "method_full":    {"alpha": a2, "beta": b2, "delta": d2, "nll": nll2, "V": V2, "Q": Q2, "time_s": t_full},
        "diag": {
            "dt": dt, "K": int(K), "S": int(S),
            "tau_eff": float(K*dt), "step_eff": float(S*dt),
            "nwin": int(nwin),
            "phase_source": "pm_volts" if using_pm else "time_ramp",
            "n_phase": int(n_phase),
            "iters_ab": int(iters_ab),
            "delta_init_grid": int(delta_init_grid),
            "lam_floor": float(lam_floor),
            "n_min": float(n_min),
        }
    }

    print("\n[Two-method timing]")
    print(f"  CURRENT (grid+IRLS, numba): {t_irls:.3f} s total, {1e3*t_irls/max(1,nwin):.3f} ms/win (incl NaNs)")
    print(f"  FULL MLE (continuous, py):  {t_full:.3f} s total, {1e3*t_full/max(1,nwin):.3f} ms/win (incl NaNs)")

    if make_plots:
        m = np.isfinite(V1) & np.isfinite(V2)

        # V(t) overlay
        fig = plt.figure(figsize=(10,4))
        plt.plot(t_centers, V1, lw=1, label="CURRENT (grid+IRLS)")
        plt.plot(t_centers, V2, lw=1, label="FULL MLE")
        plt.ylim(-0.05, 1.3 if not clip else 1.05)
        plt.xlabel("Time (s)")
        plt.ylabel("Visibility")
        plt.title(f"Visibility vs time | tau≈{K*dt*1e3:.2f} ms")
        plt.grid(True, alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()

        if np.any(m):
            dV = V1[m] - V2[m]
            _plot_with_side_hist(
                np.arange(dV.size), dV,
                xlabel="Window index (finite)",
                ylabel="ΔV = V_current − V_full",
                ylim=(np.nanpercentile(dV, 1), np.nanpercentile(dV, 99)),
                bins=60,
                title="Difference in visibility (finite windows)"
            )

            # alpha/beta/delta diffs
            da = a1[m] - a2[m]
            db = b1[m] - b2[m]
            dd = _wrap_to_pi(d1[m] - d2[m])

            _plot_with_side_hist(np.arange(da.size), da, "Window index", "Δalpha", (np.nanpercentile(da,1), np.nanpercentile(da,99)), bins=60, title="Δalpha")
            _plot_with_side_hist(np.arange(db.size), db, "Window index", "Δbeta",  (np.nanpercentile(db,1), np.nanpercentile(db,99)), bins=60, title="Δbeta")
            _plot_with_side_hist(np.arange(dd.size), dd, "Window index", "Δdelta (wrapped, rad)", (-np.pi, np.pi), bins=60, title="Δdelta (wrapped)")

    return res


# ============================================================
# Example call (adapt to your variables)
# ============================================================
ts = DNF[4]['data']['timestamps'][0:29999]
counts = DNF[4]['data']['counts'][0:29999]
pm_volts = DNF[4]['data']['pm_volts'][0:29999]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])
res = compare_two_methods_over_time(
    ts, counts,
    f_ramp=1000.0,
    tau=3e-3,
    step=None,
    pm_volts=pm_volts,     # or None to use ideal ramp
    V2pi_volts=6.86,
    n_phase=64,
    iters_ab=8,
    delta_init_grid=64,
    full_maxiter=50,
    lam_floor=1e-9,
   n_min=0.0,
   clip=False,
   make_plots=True
 )

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

# ---- bring in these from your codebase ----
# wrap_to_pi, poisson_nll, _sigmoid
# fit_alpha_beta_delta_poisson_mle_full (your scipy version)
# _mle_phasegrid_params_irls (from your numba method)
# _to_seconds, _phase_from_ramp, _phase_from_pm_volts

def compare_current_vs_fullmle(
    ts, counts,
    f_ramp, tau, step=None,
    phase_offset=0.0,
    n_min=0.0,
    # current method params
    n_phase=64, iters_ab=8, lam_floor=1e-12,
    # full MLE params
    delta_init_grid=64, maxiter=300, tol=1e-9,
    # phase source
    pm_volts=None, V2pi_volts=None,
    # plotting
    make_plots=True,
):
    t_s = _to_seconds(np.asarray(ts)).astype(float)
    y   = np.clip(np.nan_to_num(np.asarray(counts, float), nan=0.0), 0, None)
    n = min(t_s.size, y.size)
    t_s = t_s[:n]; y = y[:n]

    dt = float(np.mean(np.diff(t_s)))
    if step is None:
        step = tau/2.0
    K = max(3, int(round(float(tau)/dt)))
    S = max(1, int(round(float(step)/dt)))
    if K > n:
        raise ValueError("K > n")

    # phase model
    if pm_volts is not None:
        if V2pi_volts is None:
            raise ValueError("Need V2pi_volts if pm_volts provided.")
        pv = np.asarray(pm_volts, float)[:n]
        phi = _phase_from_pm_volts(pv, float(V2pi_volts), phase_offset=phase_offset)
    else:
        phi = _phase_from_ramp(t_s, f_ramp=f_ramp, phase_offset=phase_offset)

    nwin = (n - K)//S + 1
    centers_idx = (np.arange(nwin) * S) + (K//2)
    t_centers = t_s[centers_idx]

    # ----------------- CURRENT (numba grid+IRLS) -----------------
    cphi = np.cos(phi).astype(np.float64)
    sphi = np.sin(phi).astype(np.float64)
    deltas = np.linspace(-np.pi, np.pi, int(n_phase), endpoint=False).astype(np.float64)

    t0 = time.perf_counter()
    a1, b1, d1, nll1, V1 = _mle_phasegrid_params_irls(
        y.astype(np.float64), cphi, sphi, K, S, float(n_min), deltas,
        iters_ab=int(iters_ab), lam_floor=float(lam_floor)
    )
    t_current = time.perf_counter() - t0

    # ----------------- FULL MLE (scipy) -----------------
    a2 = np.full(nwin, np.nan)
    b2 = np.full(nwin, np.nan)
    d2 = np.full(nwin, np.nan)
    nll2 = np.full(nwin, np.nan)
    V2 = np.full(nwin, np.nan)

    t0 = time.perf_counter()
    for w in range(nwin):
        s = w*S
        e = s+K
        yw = y[s:e]
        if float(np.sum(yw)) < float(n_min):
            continue
        phi_w = phi[s:e]
        aa, bb, dd, nn = fit_alpha_beta_delta_poisson_mle_full(
            yw, phi_w,
            lam_floor=float(lam_floor),
            delta_init_grid=int(delta_init_grid),
            maxiter=int(maxiter),
            tol=float(tol)
        )
        a2[w], b2[w], d2[w], nll2[w] = aa, bb, dd, nn
        V2[w] = (bb/aa) if aa > 0 else np.nan
    t_full = time.perf_counter() - t0

    print("\n[Timing]")
    print(f"  CURRENT grid+IRLS: {t_current:.3f} s total, {1e3*t_current/max(1,nwin):.3f} ms/win")
    print(f"  FULL MLE (scipy):  {t_full:.3f} s total, {1e3*t_full/max(1,nwin):.3f} ms/win")

    if make_plots:
        m = np.isfinite(V1) & np.isfinite(V2)

        plt.figure(figsize=(10,4))
        plt.plot(t_centers, V1, lw=1, label="Current (grid+IRLS)")
        plt.plot(t_centers, V2, lw=1, label="Full MLE (scipy)")
        plt.xlabel("time (s)")
        plt.ylabel("V")
        plt.grid(True, alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()

        if np.any(m):
            dV = V1[m] - V2[m]
            plt.figure(figsize=(6,3))
            plt.hist(dV, bins=60, density=True, alpha=0.7)
            plt.xlabel("ΔV = V_current − V_full")
            plt.ylabel("PDF")
            plt.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.show()

            dd = wrap_to_pi(d1[m] - d2[m])
            plt.figure(figsize=(6,3))
            plt.hist(dd, bins=60, density=True, alpha=0.7)
            plt.xlabel("Δδ (wrapped, rad)")
            plt.ylabel("PDF")
            plt.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.show()

    return {
        "t_centers": t_centers,
        "current": {"alpha": a1, "beta": b1, "delta": d1, "nll": nll1, "V": V1, "time_s": t_current},
        "full":    {"alpha": a2, "beta": b2, "delta": d2, "nll": nll2, "V": V2, "time_s": t_full},
        "meta": {"dt": dt, "K": K, "S": S, "tau_eff": K*dt, "step_eff": S*dt, "nwin": nwin}
    }
ts = DNF[4]['data']['timestamps'][0:29999]
counts = DNF[4]['data']['counts'][0:29999]
pm_volts = DNF[4]['data']['pm_volts'][0:29999]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])
res = compare_current_vs_fullmle(
    ts, counts,
    f_ramp=1000.0, 
    tau=3e-3, 
    step=None,
    phase_offset=0.0,
    n_min=0.0,
    # current method params
    n_phase=64, iters_ab=8, lam_floor=1e-6,
    # full MLE params
    delta_init_grid=64, maxiter=50, tol=1e-6,
    # phase source
    pm_volts=pm_volts,     # or None to use ideal ramp
    V2pi_volts=6.86,
    # plotting
    make_plots=False,)

In [ ]:
ts = DNF[4]['data']['timestamps'][0:2999]
counts = DNF[4]['data']['counts'][0:2999]
pm_volts = DNF[4]['data']['pm_volts'][0:2999]
V2pi = float(DNF[4]['data']['pm_V2pi_volts'])

In [53]:
res = compare_current_vs_fullmle(
    ts, counts,
    f_ramp=1000.0, 
    tau=3e-3, 
    step=None,
    phase_offset=0.0,
    n_min=0.0,
    # current method params
    n_phase=64, iters_ab=8, lam_floor=1e-6,
    # full MLE params
    delta_init_grid=64, maxiter=300, tol=1e-6,
    # phase source
    pm_volts=pm_volts,     # or None to use ideal ramp
    V2pi_volts=6.86,
    # plotting
    make_plots=True,)


[Timing]
  CURRENT grid+IRLS: 0.092 s total, 0.046 ms/win
  FULL MLE (scipy):  725.739 s total, 363.233 ms/win
